<a href="https://colab.research.google.com/github/KayoLage/Ferramenta-SoftPipeline-INF450/blob/main/SoftPipe_Tool_INF450_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trabalho INF450 - Ferramenta didática para Soft. Pipeline**

### Exemplo usado para debug
```
loop: ld f1,0(r1)
mult f2,f1,f1
ld f3,4(r1)
mult f3,f3,f2
mult f3,f3,f1
add f3,f3,f2
sd f3,0(r1)
addi r1,r1,8
bne r1,r2,loop
```

### Exemplo simples: x = x² + 1

```
loop:
  ld f1, 0(r1)
  mult f1, f1, f1
  addi f1, f1, 1
  sd f1, 0(r1)
  addi r1, r1, 4
  bne r1,r2,loop
```

---
## Utilitários, imports, funções e classes auxiliares

### Imports

In [41]:
import re
import copy
import graphviz
import networkx as nx
import ipywidgets as widgets
from tabulate import tabulate
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import ConnectionStyle
from IPython.display import display, clear_output
from matplotlib.patches import Ellipse, FancyArrowPatch
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox

from matplotlib._pylab_helpers import Gcf
Gcf.figs.clear()

### Classes e funções auxiliares

In [42]:
def _ensure_mem_size(mem_list, idx):
    """Auxiliar para expandir a memória dinamicamente caso o índice vá além do limite atual"""
    if idx >= len(mem_list):
        mem_list.extend([float(i) for i in range(len(mem_list), idx + 1)])

class MachineConfig:
    def __init__(self, mem_size=200, word_size=4, forced_loops=30):
        self.mem_size = mem_size
        self.word_size = word_size
        self.forced_loops = forced_loops

        # Gera automaticamente r1=1, r2=2... e f1=1.0, f2=2.0... até 31
        self.reg_init = {}
        for i in range(1, 32):
            self.reg_init[f"r{i}"] = i
            self.reg_init[f"f{i}"] = float(i)

    def fresh_memory(self):
        return [float(x) for x in range(self.mem_size)]

    def fresh_registers(self):
        return dict(self.reg_init)

    def __repr__(self):
        return f"MachineConfig(mem_size={self.mem_size}, word_size={self.word_size}, forced_loops={self.forced_loops})"


class ISASimulator:
    def __init__(self, config: MachineConfig):
        self.config = config

    @staticmethod
    def _get_reg(name, R, F):
        return F.setdefault(name, 0.0) if name.startswith('f') else R.setdefault(name, 0)

    @staticmethod
    def _set_reg(name, val, R, F):
        if name.startswith('f'):
            F[name] = val
        else:
            R[name] = val

    def execute_straightline(self, instr, R, F, mem):
        """Executa UMA instrução não-branch/jump/label com auto-expansão de memória."""
        cat, ops, op = instr['category'], instr['operands'], instr['op']
        ws = self.config.word_size

        if cat == 'r_type':
            dest, s1, s2 = ops
            a, b = self._get_reg(s1, R, F), self._get_reg(s2, R, F)
            val = a * b if op == '*' else (a + b if op == '+' else a - b)
            self._set_reg(dest, val, R, F)
        elif cat == 'i_type':
            dest, src, imm = ops
            a = self._get_reg(src, R, F)
            val = a + int(imm) if op == '+' else a - int(imm)
            self._set_reg(dest, val, R, F)
        elif cat == 'load':
            dest, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)
            self._set_reg(dest, mem[idx], R, F)
        elif cat == 'store':
            src, mem_op = ops
            offset, base = MEM_RE.match(mem_op).groups()
            addr = self._get_reg(base, R, F) + int(offset)
            idx = addr // ws
            _ensure_mem_size(mem, idx)
            mem[idx] = self._get_reg(src, R, F)
        elif cat == 'mov':
            dest, src = ops
            self._set_reg(dest, self._get_reg(src, R, F), R, F)
        else:
            raise ValueError(f"categoria '{cat}' não é executável diretamente.")

    @staticmethod
    def build_label_map(instructions):
        labels = {}
        for i, instr in enumerate(instructions):
            if instr.get('type') == 'label' or instr.get('label'):
                labels[instr['label']] = i
        return labels

    def run_full_program(self, instructions, R_init=None, F_init=None, mem=None, max_steps=500_000):
        R = dict(R_init) if R_init is not None else self.config.fresh_registers()
        # FIX DEFINITIVO: Inicializa F com os valores flutuantes reais da máquina (f1=1.0, f2=2.0...)
        F = dict(F_init) if F_init is not None else {k: v for k, v in self.config.fresh_registers().items() if k.startswith('f')}
        mem = mem if mem is not None else self.config.fresh_memory()
        labels = self.build_label_map(instructions)
        pc, steps = 0, 0
        label_visits = defaultdict(int)
        n = len(instructions)

        while 0 <= pc < n and steps < max_steps:
            instr = instructions[pc]
            steps += 1

            if instr.get('type') == 'label':
                pc += 1
                continue

            if instr.get('label'):
                label_visits[instr['label']] += 1

            cat = instr['category']
            if cat == 'branch':
                s1, s2, target = instr['operands']
                a, b, op = self._get_reg(s1, R, F), self._get_reg(s2, R, F), instr['op']
                taken = {'!=': a != b, '==': a == b, '<': a < b,
                         '>': a > b, '<=': a <= b, '>=': a >= b}[op]

                if taken and target in label_visits and label_visits[target] >= self.config.forced_loops:
                    taken = False

                pc = labels[target] if taken else pc + 1

            elif cat == 'jump':
                (target,) = instr['operands']
                if target in label_visits and label_visits[target] >= self.config.forced_loops:
                    pc = pc + 1
                else:
                    pc = labels[target]
            else:
                self.execute_straightline(instr, R, F, mem)
                pc += 1

        return R, F, mem, label_visits

    @staticmethod
    def detect_induction_strides(instructions):
        strides = {}
        for instr in instructions:
            # 🔧 FIX: Usar .get() para evitar KeyError em linhas de rótulo (labels puros)
            if instr.get('category') == 'i_type':
                dest, src, imm = instr['operands']
                if dest == src and not dest.startswith('f'):
                    strides[dest] = int(imm) if instr['op'] == '+' else -int(imm)
        return strides

    @staticmethod
    def collect_written_regs(instructions):
        written = set()
        for instr in instructions:
            if instr.get('category') in ('r_type', 'i_type', 'load', 'mov'):
                written.add(instr['operands'][0])
        return written


class PipelineValidator:
    def __init__(self, original_instructions, config: MachineConfig):
        self.original_instructions = original_instructions
        self.config = config
        self.simulator = ISASimulator(config)

    def compute_reference(self):
        _, _, mem_final, label_visits = self.simulator.run_full_program(self.original_instructions)
        strides = self.simulator.detect_induction_strides(self.original_instructions)
        total_iterations = max(label_visits.values()) if label_visits else 1
        return mem_final, strides, total_iterations

    def simulate_pipeline(self, stages_user, strides, total_iterations):
        ws = self.config.word_size
        mem_pipe = self.config.fresh_memory()
        # FIX DEFINITIVO: Inicializa F_pipe com os valores flutuantes corretos do estado inicial
        F_pipe = {k: v for k, v in self.config.reg_init.items() if k.startswith('f')}
        max_lvl = max(stages_user.keys()) if stages_user else 0
        total_cycles = total_iterations + max_lvl

        for cycle in range(total_cycles):
            F_snap = dict(F_pipe)
            mem_snap = list(mem_pipe)
            pending_F, pending_mem = {}, {}

            for lvl in sorted(stages_user.keys(), reverse=True):
                it = cycle - lvl
                if not (0 <= it < total_iterations):
                    continue
                R_it = {reg: self.config.reg_init.get(reg, 0) + it * step for reg, step in strides.items()}
                for instr in stages_user[lvl]:
                    cat, ops, op = instr['category'], instr['operands'], instr['op']
                    if cat == 'load':
                        dest, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws

                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        val = mem_snap[idx] if idx < len(mem_snap) else float(idx)
                        pending_F[dest] = val
                    elif cat == 'store':
                        src, mem_op = ops
                        offset, base = MEM_RE.match(mem_op).groups()
                        addr = R_it.get(base, 0) + int(offset)
                        idx = addr // ws
                        if idx >= len(mem_pipe):
                            _ensure_mem_size(mem_pipe, idx)
                        pending_mem[idx] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'mov':
                        dest, src = ops
                        pending_F[dest] = F_snap.get(src, pending_F.get(src, 0.0))
                    elif cat == 'r_type':
                        dest, s1, s2 = ops
                        a = F_snap.get(s1, pending_F.get(s1, 0.0))
                        b = F_snap.get(s2, pending_F.get(s2, 0.0))
                        pending_F[dest] = a * b if op == '*' else (a + b if op == '+' else a - b)
                    elif cat == 'i_type':
                        dest, src, imm = ops
                        a = F_snap.get(src, pending_F.get(src, 0.0)) if src.startswith('f') else R_it.get(src, 0)
                        pending_F[dest] = a + int(imm) if op == '+' else a - int(imm)

            F_pipe.update(pending_F)
            for idx, val in pending_mem.items():
                mem_pipe[idx] = val

        return mem_pipe

    def validate(self, user_canvas_nodes):
        if not user_canvas_nodes:
            return {"error": "O canvas está vazio! Monte um grafo antes de validar."}
        if not self.original_instructions:
            return {"error": "Nenhum programa original carregado para servir de referência."}

        try:
            mem_gabarito, strides, total_iterations = self.compute_reference()
        except Exception as e:
            return {"error": f"Erro executando o programa original: {e}"}

        stages_user = defaultdict(list)
        for item in user_canvas_nodes:
            stages_user[item['level']].append(parse_line(item['text'], 0))

        try:
            mem_pipe = self.simulate_pipeline(stages_user, strides, total_iterations)
        except Exception as e:
            return {"error": f"Erro na simulação do pipeline: {e}"}

        return {
            "success": mem_gabarito == mem_pipe,
            "mem_gabarito": mem_gabarito,
            "mem_pipe": mem_pipe,
            "strides": strides,
            "total_iterations": total_iterations,
            "error": None,
        }

### Classe para Tomasulo - Escalonamento Dinâmico

In [61]:
# @title Pipeline Processador com Escalonamento Dinâmico e In-Order

# BLOCO 1: ESTRUTURAS BÁSICAS E PARSER
class SyntaxErrorTomasulo(Exception):
    pass

class Instruction:
    def __init__(self, raw_line, pc_address, global_id):
        self.raw_line = raw_line.strip()
        self.pc = pc_address
        self.id = global_id

        self.op = None
        self.dest = None
        self.src1 = None
        self.src2 = None
        self.imm = None
        self.target_address = None

        self.is_float_write = False
        self.is_int_write = False

        self.cycle_fetch_start = None
        self.cycle_fetch = None
        self.cycle_issue = None
        self.cycle_exec_start = None
        self.cycle_exec_end = None
        self.cycle_write = None

        self.flushed = False
        self.cloned = False

        self.val1 = None
        self.val2 = None
        self.result = 0.0

class ReservationStation:
    def __init__(self, name):
        self.name = name
        self.busy = False
        self.inst = None
        self.Vj = self.Vk = self.Qj = self.Qk = None
        self.time_remaining = 0

    def clear(self):
        self.busy = False
        self.inst = None
        self.Vj = self.Vk = self.Qj = self.Qk = None
        self.time_remaining = 0

class Parser:
    @staticmethod
    def validate_reg(reg, expected_type=None):
        match = re.match(r'^([fr])([0-9]+)$', reg)
        if not match: raise SyntaxErrorTomasulo(f"Registo inválido '{reg}'. Use f1-f20 ou r1-r20.")
        r_type, r_num = match.groups()
        if not (1 <= int(r_num) <= 20): raise SyntaxErrorTomasulo(f"Registo '{reg}' fora dos limites (1-20).")
        if expected_type and r_type != expected_type: raise SyntaxErrorTomasulo(f"Esperava-se tipo '{expected_type}', encontrou '{reg}'.")
        return reg

    @classmethod
    def parse_registry_initialization(cls, line):
        content = line.replace('.reg', '', 1).strip()
        if not content: raise SyntaxErrorTomasulo("Diretiva '.reg' vazia.")
        initializations = {}
        for part in content.split(';'):
            part = part.strip()
            if not part: continue
            if '=' not in part: raise SyntaxErrorTomasulo(f"Formato inválido: '{part}'. Use 'reg = valor'.")
            reg, val_str = part.split('=', 1)
            reg = cls.validate_reg(reg.strip().lower())
            try:
                initializations[reg] = float(val_str.strip()) if reg.startswith('f') else int(val_str.strip())
            except ValueError:
                raise SyntaxErrorTomasulo(f"Valor inválido '{val_str}' para '{reg}'.")
        return initializations

    @classmethod
    def parse_memory_initialization(cls, line):
        content = line.replace('.mem', '', 1).strip()
        if not content: raise SyntaxErrorTomasulo("Diretiva '.mem' vazia.")
        initializations = {}
        for part in content.split(';'):
            part = part.strip()
            if not part: continue
            if '=' not in part: raise SyntaxErrorTomasulo(f"Formato inválido: '{part}'. Use 'endereco = valor'.")
            addr_str, val_str = part.split('=', 1)
            try: addr = int(addr_str.strip())
            except ValueError: raise SyntaxErrorTomasulo(f"Endereço inválido '{addr_str}'.")
            if addr < 0 or addr % 4 != 0: raise SyntaxErrorTomasulo(f"Endereço '{addr}' inválido. Deve ser positivo e múltiplo de 4.")
            try: initializations[addr] = float(val_str.strip())
            except ValueError: raise SyntaxErrorTomasulo(f"Valor inválido '{val_str}' para memória.")
        return initializations

    @classmethod
    def parse_line(cls, line, pc_address, global_id, labels_map=None):
        line = line.split('#')[0].strip()
        if not line: return None

        comma_count = line.count(',')
        inst = Instruction(line, pc_address, global_id)
        parts = line.split(maxsplit=1)
        original_op = parts[0]

        if ',' in original_op: raise SyntaxErrorTomasulo(f"Vírgula colada à operação: '{original_op}'")
        if original_op != original_op.lower(): raise SyntaxErrorTomasulo(f"Instrução deve ser minúscula (ex: '{original_op.lower()}').")

        inst.op = original_op.lower()
        operands = [op.strip() for op in parts[1].replace('(', ',').replace(')', '').split(',')] if len(parts) > 1 else []
        operands = [op for op in operands if op]

        def get_int(val, name):
            try: return int(val)
            except ValueError: raise SyntaxErrorTomasulo(f"[{inst.op}] '{name}' deve ser inteiro. Encontrou: '{val}'")

        if inst.op in ['multf', 'addf', 'add', 'addi', 'bne']:
            if comma_count != 2 or len(operands) != 3: raise SyntaxErrorTomasulo(f"Sintaxe incorreta para '{inst.op}'.")
            if inst.op in ['multf', 'addf']:
                inst.dest, inst.src1, inst.src2 = cls.validate_reg(operands[0], 'f'), cls.validate_reg(operands[1], 'f'), cls.validate_reg(operands[2], 'f')
                inst.is_float_write = True
            elif inst.op == 'add':
                inst.dest, inst.src1, inst.src2 = cls.validate_reg(operands[0], 'r'), cls.validate_reg(operands[1], 'r'), cls.validate_reg(operands[2], 'r')
                inst.is_int_write = True
            elif inst.op == 'addi':
                inst.dest, inst.src1, inst.imm = cls.validate_reg(operands[0], 'r'), cls.validate_reg(operands[1], 'r'), get_int(operands[2], 'imediato')
                inst.is_int_write = True
            elif inst.op == 'bne':
                inst.src1, inst.src2 = cls.validate_reg(operands[0], 'r'), cls.validate_reg(operands[1], 'r')
                target_str = operands[2]
                if labels_map and target_str in labels_map: inst.target_address = labels_map[target_str]
                else:
                    try: inst.target_address = int(target_str)
                    except ValueError: raise SyntaxErrorTomasulo(f"Alvo inválido/label não declarada: '{target_str}'")
                if inst.target_address % 4 != 0: raise SyntaxErrorTomasulo("Destino do branch deve ser múltiplo de 4.")

        elif inst.op == 'movf':
            if comma_count != 1 or len(operands) != 2: raise SyntaxErrorTomasulo(f"Sintaxe incorreta para '{inst.op}'.")
            inst.dest, inst.src1 = cls.validate_reg(operands[0], 'f'), cls.validate_reg(operands[1], 'f')
            inst.is_float_write = True

        elif inst.op in ['ld', 'sd']:
            if comma_count != 1 or len(operands) != 3: raise SyntaxErrorTomasulo(f"Sintaxe incorreta para '{inst.op}'.")
            if inst.op == 'ld':
                inst.dest, inst.imm, inst.src1 = cls.validate_reg(operands[0], 'f'), get_int(operands[1], 'offset'), cls.validate_reg(operands[2], 'r')
                inst.is_float_write = True
            elif inst.op == 'sd':
                inst.src2, inst.imm, inst.src1 = cls.validate_reg(operands[0], 'f'), get_int(operands[1], 'offset'), cls.validate_reg(operands[2], 'r')
        else:
            raise SyntaxErrorTomasulo(f"Instrução desconhecida: '{inst.op}'")
        return inst

# BLOCO 2A: PROCESSADOR TOMASULO
class TomasuloCore:
    def __init__(self, instructions_list, config, initial_regs=None, initial_mem=None):
        self.instructions = instructions_list
        self.config = config
        self.cycle = 0
        self.pc = 0
        self.flush_fetch_this_cycle = False

        self.regs_f = {f"f{i}": 0.0 for i in range(1, 21)}
        self.regs_r = {f"r{i}": 0 for i in range(1, 21)}
        self.regs_status_f = {f"f{i}": None for i in range(1, 21)}
        self.regs_status_r = {f"r{i}": None for i in range(1, 21)}

        if initial_regs:
            for reg, val in initial_regs.items():
                if reg.startswith('f'): self.regs_f[reg] = float(val)
                else: self.regs_r[reg] = int(val)

        self.memory = {i * 4: float(i) for i in range(100)}
        self.accessed_memory = set()
        if initial_mem:
            for addr, val in initial_mem.items():
                self.memory[addr] = val
                self.accessed_memory.add(addr)

        self.fetch_queue = []
        self.stations = {
            'INT_ALU': [ReservationStation(f'INT{i+1}') for i in range(config.get('num_int', 2))],
            'FLOAT_ADD': [ReservationStation(f'FADD{i+1}') for i in range(config.get('num_add', 2))],
            'MULT': [ReservationStation(f'MULT{i+1}') for i in range(config.get('num_mult', 2))],
            'LS': [ReservationStation(f'LS{i+1}') for i in range(config.get('num_ls', 2))]
        }
        self.history = []

    def save_state(self):
        state = {
            'cycle': self.cycle, 'pc': self.pc, 'flush_fetch_this_cycle': self.flush_fetch_this_cycle,
            'regs_f': self.regs_f, 'regs_r': self.regs_r,
            'regs_status_f': self.regs_status_f, 'regs_status_r': self.regs_status_r,
            'memory': self.memory, 'fetch_queue': self.fetch_queue,
            'accessed_memory': self.accessed_memory, 'stations': self.stations,
            'instructions': self.instructions
        }
        self.history.append(copy.deepcopy(state))

    def step(self):
        self.save_state()
        self.cycle += 1
        self._stage_writeback()
        self._stage_execute()
        self._stage_decode_issue()
        self._stage_fetch()

    def _stage_writeback(self):
        f_done, i_done = False, False
        for rs_list in self.stations.values():
            for rs in rs_list:
                if rs.busy and rs.time_remaining == 0 and rs.inst.cycle_exec_end is not None and rs.inst.cycle_write is None:
                    inst = rs.inst
                    if inst.is_float_write and f_done: continue
                    if inst.is_int_write and i_done: continue

                    res = self._calc(rs)
                    if inst.op == 'sd':
                        addr = inst.imm + int(rs.Vj)
                        self.accessed_memory.add(addr)
                        self.memory[addr] = float(rs.Vk)
                    else:
                        st_dict = self.regs_status_f if inst.dest.startswith('f') else self.regs_status_r
                        if st_dict[inst.dest] == rs.name:
                            if inst.dest.startswith('f'): self.regs_f[inst.dest] = float(res)
                            else: self.regs_r[inst.dest] = int(res)
                            st_dict[inst.dest] = None
                        if inst.is_float_write: f_done = True
                        if inst.is_int_write: i_done = True

                        for r_lst in self.stations.values():
                            for o_rs in r_lst:
                                if o_rs.busy:
                                    if o_rs.Qj == rs.name: o_rs.Vj, o_rs.Qj = res, None
                                    if o_rs.Qk == rs.name: o_rs.Vk, o_rs.Qk = res, None
                    inst.cycle_write = self.cycle
                    rs.clear()

    def _calc(self, rs):
        vj, vk = rs.Vj or 0, rs.Vk or 0
        if rs.inst.op == 'ld':
            addr = rs.inst.imm + int(vj)
            self.accessed_memory.add(addr)
            return self.memory.get(addr, 0.0)
        elif rs.inst.op in ['addf', 'add']: return vj + vk
        elif rs.inst.op == 'movf': return vj
        elif rs.inst.op == 'addi': return vj + rs.inst.imm
        elif rs.inst.op == 'multf': return vj * vk
        return 0

    def _stage_execute(self):
        for rs_list in self.stations.values():
            for rs in rs_list:
                if rs.busy and rs.time_remaining > 0 and rs.Qj is None and rs.Qk is None:
                    if rs.inst.cycle_exec_start is None:
                        rs.inst.cycle_exec_start = self.cycle
                        rs.inst.cycle_exec_end = self.cycle + rs.time_remaining - 1
                    rs.time_remaining -= 1

    def _stage_decode_issue(self):
        ways = self.config.get('n_way', 1)
        emitted = 0
        while self.fetch_queue and emitted < ways:
            inst = self.fetch_queue[0]
            if inst.op == 'bne':
                st1 = self.regs_status_r[inst.src1]
                st2 = self.regs_status_r[inst.src2]
                if st1 is not None or st2 is not None: break
                inst.cycle_issue = self.cycle
                if self.regs_r[inst.src1] != self.regs_r[inst.src2]:
                    self.pc = inst.target_address
                    for f in self.fetch_queue[1:]: f.flushed = True
                    self.fetch_queue.clear()
                    for i in self.instructions:
                        if i.cloned and i.cycle_fetch is None: i.flushed = True
                    self.flush_fetch_this_cycle = True
                else: self.fetch_queue.pop(0)
                emitted += 1
                continue

            rtype = 'INT_ALU' if inst.op in ['add','addi'] else 'FLOAT_ADD' if inst.op in ['addf', 'movf'] else 'MULT' if inst.op == 'multf' else 'LS'
            free_rs = next((r for r in self.stations[rtype] if not r.busy), None)
            if not free_rs: break

            free_rs.busy, free_rs.inst = True, inst
            inst.cycle_issue = self.cycle

            if inst.op in ['ld', 'sd']: free_rs.time_remaining = self.config.get('time_ls', 2)
            elif inst.op in ['add', 'addi']: free_rs.time_remaining = self.config.get('time_int', 1)
            elif inst.op == 'movf': free_rs.time_remaining = 1 # Latência de 1 ciclo para movimentação simples
            elif inst.op == 'addf': free_rs.time_remaining = self.config.get('time_add', 3)
            elif inst.op == 'multf': free_rs.time_remaining = self.config.get('time_mult', 5)

            if inst.src1:
                st = self.regs_status_f[inst.src1] if inst.src1.startswith('f') else self.regs_status_r[inst.src1]
                if st is None: free_rs.Vj = self.regs_f[inst.src1] if inst.src1.startswith('f') else self.regs_r[inst.src1]
                else: free_rs.Qj = st
            if inst.src2:
                st = self.regs_status_f[inst.src2] if inst.src2.startswith('f') else self.regs_status_r[inst.src2]
                if st is None: free_rs.Vk = self.regs_f[inst.src2] if inst.src2.startswith('f') else self.regs_r[inst.src2]
                else: free_rs.Qk = st

            if inst.dest and inst.op != 'sd':
                st_dict = self.regs_status_f if inst.dest.startswith('f') else self.regs_status_r
                st_dict[inst.dest] = free_rs.name

            self.fetch_queue.pop(0)
            emitted += 1

    def _stage_fetch(self):
        if self.flush_fetch_this_cycle:
            self.flush_fetch_this_cycle = False
            return
        ways = self.config.get('n_way', 1)
        space = ways - len(self.fetch_queue)
        tpc = self.pc
        for _ in range(ways):
            orig = next((i for i in self.instructions if i.pc == tpc and not i.cloned), None)
            if not orig: break
            bi = next((i for i in self.instructions if i.pc == tpc and i.cycle_fetch is None and not i.flushed), None)
            if not bi:
                bi = copy.deepcopy(orig)
                bi.cloned, bi.flushed = True, False
                bi.cycle_fetch_start = self.cycle
                bi.cycle_fetch = None
                bi.cycle_issue = None
                bi.cycle_exec_start = None
                bi.cycle_exec_end = None
                bi.cycle_write = None
                bi.val1 = None
                bi.val2 = None
                bi.result = 0.0
                self.instructions.append(bi)
            elif bi.cycle_fetch_start is None: bi.cycle_fetch_start = self.cycle
            tpc += 4

        fetched = 0
        while fetched < space:
            bi = next((i for i in self.instructions if i.pc == self.pc and i.cycle_fetch is None and not i.flushed), None)
            if bi:
                bi.cycle_fetch = self.cycle
                self.fetch_queue.append(bi)
                self.pc += 4
                fetched += 1
            else: break


# BLOCO 2B: PROCESSADOR SIMPLES (IN-ORDER)
class SimpleCore:
    def __init__(self, instructions_list, config, initial_regs=None, initial_mem=None):
        self.instructions = instructions_list
        self.config = config
        self.cycle = 0
        self.pc = 0
        self.flush_fetch_this_cycle = False

        self.regs_f = {f"f{i}": 0.0 for i in range(1, 21)}
        self.regs_r = {f"r{i}": 0 for i in range(1, 21)}
        self.regs_status_f = {f"f{i}": None for i in range(1, 21)}
        self.regs_status_r = {f"r{i}": None for i in range(1, 21)}

        if initial_regs:
            for reg, val in initial_regs.items():
                if reg.startswith('f'): self.regs_f[reg] = float(val)
                else: self.regs_r[reg] = int(val)

        self.memory = {i * 4: float(i) for i in range(100)}
        self.accessed_memory = set()
        if initial_mem:
            for addr, val in initial_mem.items():
                self.memory[addr] = val
                self.accessed_memory.add(addr)

        self.fetch_queue = []
        self.executing = []
        self.ready_to_write = []

        self.execution_units = {
            'INT_ALU': 1, 'FLOAT_ADD': 1, 'MULT': 1, 'LS': 1
        }
        self.history = []

    def save_state(self):
        state = {
            'cycle': self.cycle, 'pc': self.pc, 'flush_fetch_this_cycle': self.flush_fetch_this_cycle,
            'regs_f': self.regs_f, 'regs_r': self.regs_r,
            'regs_status_f': self.regs_status_f, 'regs_status_r': self.regs_status_r,
            'memory': self.memory, 'fetch_queue': self.fetch_queue,
            'executing': self.executing, 'ready_to_write': self.ready_to_write,
            'execution_units': self.execution_units, 'accessed_memory': self.accessed_memory,
            'instructions': self.instructions
        }
        self.history.append(copy.deepcopy(state))

    def step(self):
        self.save_state()
        self.cycle += 1
        self._stage_writeback()
        self._stage_execute()
        self._stage_decode_issue()
        self._stage_fetch()

    def _stage_writeback(self):
        f_done, i_done = False, False
        for inst in self.ready_to_write[:]:
            if inst.is_float_write and f_done: continue
            if inst.is_int_write and i_done: continue

            if inst.op == 'sd':
                addr = inst.imm + int(inst.val1)
                self.accessed_memory.add(addr)
                self.memory[addr] = float(inst.val2)
            else:
                st_dict = self.regs_status_f if inst.dest.startswith('f') else self.regs_status_r
                if inst.dest.startswith('f'): self.regs_f[inst.dest] = float(inst.result)
                else: self.regs_r[inst.dest] = int(inst.result)
                if st_dict[inst.dest] == inst: st_dict[inst.dest] = None

            inst.cycle_write = self.cycle
            self.ready_to_write.remove(inst)
            if inst.is_float_write: f_done = True
            if inst.is_int_write: i_done = True

    def _stage_execute(self):
        for inst in self.executing[:]:
            if inst.cycle_exec_start is None:
                inst.cycle_exec_start = self.cycle
                inst.cycle_exec_end = self.cycle + inst.time_remaining - 1

            inst.time_remaining -= 1
            if inst.time_remaining == 0:
                v1 = inst.val1 if inst.val1 is not None else 0
                v2 = inst.val2 if inst.val2 is not None else 0
                if inst.op == 'ld':
                    addr = inst.imm + int(v1)
                    self.accessed_memory.add(addr)
                    inst.result = self.memory.get(addr, 0.0)
                elif inst.op in ['add', 'addf']: inst.result = v1 + v2
                elif inst.op == 'movf': inst.result = v1
                elif inst.op == 'addi': inst.result = v1 + inst.imm
                elif inst.op == 'multf': inst.result = v1 * v2

                self.executing.remove(inst)
                self.ready_to_write.append(inst)
                rtype = 'INT_ALU' if inst.op in ['add','addi'] else 'FLOAT_ADD' if inst.op in ['addf', 'movf'] else 'MULT' if inst.op == 'multf' else 'LS'
                self.execution_units[rtype] += 1

    def _get_fwd_val(self, reg):
        st = self.regs_status_f[reg] if reg.startswith('f') else self.regs_status_r[reg]
        if st is None: return self.regs_f[reg] if reg.startswith('f') else self.regs_r[reg]
        return st.result

    def _stage_decode_issue(self):
        ways = self.config.get('n_way', 1)
        emitted = 0
        while self.fetch_queue and emitted < ways:
            inst = self.fetch_queue[0]

            stalled = False
            for src in [inst.src1, inst.src2]:
                if src:
                    st = self.regs_status_f[src] if src.startswith('f') else self.regs_status_r[src]
                    if stalled or (st is not None and (st.cycle_exec_end is None or st.cycle_exec_end > self.cycle)):
                        stalled = True; break
            if stalled: break

            if inst.op == 'bne':
                inst.val1 = self._get_fwd_val(inst.src1)
                inst.val2 = self._get_fwd_val(inst.src2)
                inst.cycle_issue = self.cycle
                if inst.val1 != inst.val2:
                    self.pc = inst.target_address
                    for f in self.fetch_queue[1:]: f.flushed = True
                    self.fetch_queue.clear()
                    for i in self.instructions:
                        if i.cloned and i.cycle_fetch is None: i.flushed = True
                    self.flush_fetch_this_cycle = True
                else: self.fetch_queue.pop(0)
                emitted += 1
                continue

            rtype = 'INT_ALU' if inst.op in ['add','addi'] else 'FLOAT_ADD' if inst.op in ['addf', 'movf'] else 'MULT' if inst.op == 'multf' else 'LS'
            if self.execution_units[rtype] <= 0: break

            time_rem = 0
            if inst.op in ['ld', 'sd']: time_rem = self.config.get('time_ls', 2)
            elif inst.op in ['add', 'addi']: time_rem = self.config.get('time_int', 1)
            elif inst.op == 'movf': time_rem = 1
            elif inst.op == 'addf': time_rem = self.config.get('time_add', 3)
            elif inst.op == 'multf': time_rem = self.config.get('time_mult', 5)

            projected_wb_cycle = self.cycle + time_rem + 1

            if inst.dest and inst.op != 'sd':
                st = self.regs_status_f[inst.dest] if inst.dest.startswith('f') else self.regs_status_r[inst.dest]
                if st is not None:
                    st_wb = (st.cycle_exec_end + 1) if st.cycle_exec_end is not None else (st.cycle_issue + st.time_remaining + 1)
                    if projected_wb_cycle <= st_wb:
                        break

            wb_conflict = False
            for ex_inst in self.executing:
                ex_wb_cycle = (ex_inst.cycle_exec_end + 1) if ex_inst.cycle_exec_end is not None else (ex_inst.cycle_issue + ex_inst.time_remaining + 1)
                if ex_wb_cycle == projected_wb_cycle:
                    if inst.is_float_write and ex_inst.is_float_write: wb_conflict = True; break
                    if inst.is_int_write and ex_inst.is_int_write: wb_conflict = True; break

            if wb_conflict: break

            if inst.src1: inst.val1 = self._get_fwd_val(inst.src1)
            if inst.src2: inst.val2 = self._get_fwd_val(inst.src2)

            self.execution_units[rtype] -= 1
            inst.cycle_issue = self.cycle
            inst.time_remaining = time_rem

            if inst.dest and inst.op != 'sd':
                st_dict = self.regs_status_f if inst.dest.startswith('f') else self.regs_status_r
                st_dict[inst.dest] = inst

            self.executing.append(inst)
            self.fetch_queue.pop(0)
            emitted += 1

    def _stage_fetch(self):
        if self.flush_fetch_this_cycle:
            self.flush_fetch_this_cycle = False
            return
        ways = self.config.get('n_way', 1)
        space = ways - len(self.fetch_queue)
        tpc = self.pc
        for _ in range(ways):
            orig = next((i for i in self.instructions if i.pc == tpc and not i.cloned), None)
            if not orig: break
            bi = next((i for i in self.instructions if i.pc == tpc and i.cycle_fetch is None and not i.flushed), None)
            if not bi:
                bi = copy.deepcopy(orig)
                bi.cloned, bi.flushed = True, False
                bi.cycle_fetch_start = self.cycle
                bi.cycle_fetch = None
                bi.cycle_issue = None
                bi.cycle_exec_start = None
                bi.cycle_exec_end = None
                bi.cycle_write = None
                bi.val1 = None
                bi.val2 = None
                bi.result = 0.0
                self.instructions.append(bi)
            elif bi.cycle_fetch_start is None: bi.cycle_fetch_start = self.cycle
            tpc += 4

        fetched = 0
        while fetched < space:
            bi = next((i for i in self.instructions if i.pc == self.pc and i.cycle_fetch is None and not i.flushed), None)
            if bi:
                bi.cycle_fetch = self.cycle
                self.fetch_queue.append(bi)
                self.pc += 4
                fetched += 1
            else: break


# BLOCO 3: INTERFACE COMPARATIVA LADO A LADO
class SimulatorController:
    def __init__(self):
        self.core_a = None
        self.core_b = None
        self.is_tomasulo_a = True
        self.is_tomasulo_b = True
        self.build_setup_ui()

    def get_scenario_stream(self, mode):
        """Retorna o fluxo de instruções baseado no cenário selecionado."""
        if mode == 'original':
            return [
                "loop: ld f1, 0(r1)",
                "ld f3, 4(r1)",
                "multf f2, f1, f1",
                "multf f3, f3, f2",
                "multf f3, f3, f1",
                "addf f3, f3, f2",
                "sd f3, 0(r1)",
                "addi r1, r1, 8",
                "bne r1, r2, loop"
            ]
        elif mode == 'gabarito':
            return [
                "loop: sd f11, 0(r1)",
                "addf f11, f10, f9",
                "movf f9, f7",
                "multf f10, f8, f6",
                "movf f6, f4",
                "movf f7, f2",
                "multf f8, f5, f2",
                "movf f4, f1",
                "multf f2, f1, f1",
                "movf f5, f3",
                "ld f1, 0(r1)",
                "ld f3, 4(r1)",
                "addi r1, r1, 8",
                "bne r1, r2, loop"
            ]
        elif mode == 'canvas':
            user_canvas = globals().get('user_canvas_nodes', None)
            if not user_canvas:
                return None

            # Ordena decrescentemente pelo nível do nó (N=5 desce até N=0)
            canvas_nodes_sorted = sorted(user_canvas, key=lambda x: x.get('level', 0), reverse=True)
            stream = []
            for item in canvas_nodes_sorted:
                text = item.get('text', '').strip()
                # Compatibiliza termos genéricos adicionados no canvas para a sintaxe do parser
                text = text.replace('mult ', 'multf ').replace('add ', 'addf ').replace('mov ', 'movf ')
                stream.append(text)

            if stream:
                stream[0] = f"loop: {stream[0]}"

            stream.extend([
                "addi r1, r1, 8",
                "bne r1, r2, loop"
            ])
            return stream
        return []

    def build_config_panel(self, title):
        is_a = "A" in title
        mode = widgets.Dropdown(
            options=['Tomasulo (Fora de Ordem)', 'Pipeline Simples (Em Ordem)'],
            value='Tomasulo (Fora de Ordem)' if is_a else 'Pipeline Simples (Em Ordem)',
            description='Arquitetura:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%')
        )
        n_way = widgets.Dropdown(options=[1, 2, 3, 4], value=1 if is_a else 2, description='Superescalar (Way):', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%'))

        rs_int = widgets.BoundedIntText(value=2, min=1, max=10, description='Qtd. Int ALU:', layout=widgets.Layout(width='48%'))
        rs_add = widgets.BoundedIntText(value=2, min=1, max=10, description='Qtd. Float Add:', layout=widgets.Layout(width='48%'))
        rs_mult = widgets.BoundedIntText(value=2, min=1, max=10, description='Qtd. Mult:', layout=widgets.Layout(width='48%'))
        rs_ls = widgets.BoundedIntText(value=2, min=1, max=10, description='Qtd. Ld/Sd:', layout=widgets.Layout(width='48%'))

        rs_container = widgets.VBox([
            widgets.HTML("<i style='font-size:11px; color:gray;'>Estações de Reserva (Tomasulo):</i>"),
            widgets.HBox([rs_int, rs_add]), widgets.HBox([rs_mult, rs_ls])
        ], layout=widgets.Layout(width='100%'))

        rs_container.layout.display = 'flex' if is_a else 'none'

        def on_mode_change(change):
            if 'Tomasulo' in change['new']: rs_container.layout.display = 'flex'
            else: rs_container.layout.display = 'none'
        mode.observe(on_mode_change, names='value')

        time_int = widgets.BoundedIntText(value=1, min=1, max=20, description='Ciclos Int', layout=widgets.Layout(width='48%'))
        time_add = widgets.BoundedIntText(value=3, min=1, max=20, description='Ciclos Fl. Add', layout=widgets.Layout(width='48%'))
        time_mult = widgets.BoundedIntText(value=5, min=1, max=20, description='Ciclos Mult', layout=widgets.Layout(width='48%'))
        time_ls = widgets.BoundedIntText(value=2, min=1, max=20, description='Ciclos Ld/Sd', layout=widgets.Layout(width='48%'))

        box = widgets.VBox([
            widgets.HTML(f"<h4 style='color:inherit; margin-top:10px; border-bottom:1px solid rgba(128,128,128,0.3); padding-bottom:5px;'>{title}</h4>"),
            mode, n_way,
            rs_container,
            widgets.HTML("<i style='font-size:11px; color:gray;'>Latências das Unidades:</i>"),
            widgets.HBox([time_int, time_add]), widgets.HBox([time_mult, time_ls])
        ], layout=widgets.Layout(width='48%', padding='10px', border='1px solid rgba(128,128,128,0.4)', border_radius='4px'))

        return {'box': box, 'mode': mode, 'n_way': n_way, 'rs_int': rs_int, 'rs_add': rs_add, 'rs_mult': rs_mult, 'rs_ls': rs_ls, 'time_int': time_int, 'time_add': time_add, 'time_mult': time_mult, 'time_ls': time_ls}

    def build_setup_ui(self):
        self.out = widgets.Output()

        # Dropdown Principal de Seleção de Cenários
        self.mode_stream = widgets.Dropdown(
            options=[
                ('Código Original', 'original'),
                ('Software Pipelining (Gabarito)', 'gabarito'),
                ('Software Pipelining (Usuário)', 'canvas')
            ],
            value='original',
            description='Escalonamento:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%', margin='5px 0px')
        )

        # Caixa de prévia do código que será simulado
        self.code_preview = widgets.HTML(value="")

        self.panel_a = self.build_config_panel("Configuração da Máquina A")
        self.panel_b = self.build_config_panel("Configuração da Máquina B")

        self.btn_start = widgets.Button(description="Iniciar Comparação Lado a Lado", button_style='primary', layout=widgets.Layout(width='100%', margin='15px 0px'))
        self.btn_start.on_click(self.start_simulation)
        self.error_html = widgets.HTML(value="")

        # Configura callback de atualização da prévia quando mudar de cenário
        def update_preview(change=None):
            mode = self.mode_stream.value
            stream = self.get_scenario_stream(mode)
            if stream is None:
                self.code_preview.value = (
                    "<div style='border: 1px dashed #f59e0b; padding: 10px; border-radius: 4px; background: rgba(245, 158, 11, 0.05); font-family: monospace;'>"
                    "<b style='color: #d97706;'>⚠️ Grafo do Usuário Não Encontrado</b><br>"
                    "Por favor, monte e salve o grafo no Canvas interativo do exercício antes de tentar simular essa opção."
                    "</div>"
                )
            else:
                code_html = "<br>".join([f"&nbsp;&nbsp;&nbsp;&nbsp;{line}" if not line.startswith("loop:") else line for line in stream])
                self.code_preview.value = (
                    f"<div style='border: 1px solid rgba(128,128,128,0.3); padding: 10px; border-radius: 4px; background: rgba(128,128,128,0.05); font-family: Courier, monospace; font-size: 13px; color: inherit;'>"
                    f"<b style='color: #f59e0b;'>Fluxo de Instruções Atual:</b><br><br>{code_html}</div>"
                )

        self.mode_stream.observe(update_preview, names='value')
        update_preview() # Inicialização

        display(HTML("<h2 style='margin-bottom:0px; color:inherit;'>Simulador Multiarquitetura</h2><p style='color:gray; font-size:12px;'>Compare o comportamento do loop em duas arquiteturas distintas</p>"))
        display(self.mode_stream)
        display(self.code_preview)
        display(widgets.HBox([self.panel_a['box'], self.panel_b['box']], layout=widgets.Layout(justify_content='space-between')))
        display(self.btn_start, self.error_html, self.out)

    def create_core_helper(self, panel, insts, regs, mem):
        is_tomasulo = ('Tomasulo' in panel['mode'].value)
        cfg = {
            'n_way': panel['n_way'].value,
            'num_int': panel['rs_int'].value, 'num_add': panel['rs_add'].value,
            'num_mult': panel['rs_mult'].value, 'num_ls': panel['rs_ls'].value,
            'time_int': panel['time_int'].value, 'time_add': panel['time_add'].value,
            'time_mult': panel['time_mult'].value, 'time_ls': panel['time_ls'].value
        }
        if is_tomasulo:
            return TomasuloCore(insts, cfg, initial_regs=regs, initial_mem=mem), True
        return SimpleCore(insts, cfg, initial_regs=regs, initial_mem=mem), False

    def start_simulation(self, b):
        self.error_html.value = ""
        with self.out: clear_output()

        mode = self.mode_stream.value
        stream_raw = self.get_scenario_stream(mode)

        if stream_raw is None:
            self.error_html.value = (
                "<div style='padding: 10px; background-color: #fef3c7; border-left: 4px solid #d97706; color: #92400e; border-radius: 4px; font-family: sans-serif; margin-bottom: 10px;'>"
                "⚠️ <b>Grafo do Usuário não encontrado!</b><br>"
                "Por favor, certifique-se de preencher o Canvas interativo antes de tentar simular esta opção."
                "</div>"
            )
            return

        # Banco de Registradores pré-configurado
        # R1 inicia em 0 e R2 em 16 (forçando 2 iterações completas do loop de passo 8)
        initial_regs = {'r1': 0, 'r2': 16}
        initial_mem = {0: 1.5, 4: 2.0, 8: 2.5, 12: 3.0}
        labels_map = {'loop': 0}

        stream_objs = []
        for i, raw in enumerate(stream_raw):
            clean_raw = raw
            if "loop:" in raw:
                clean_raw = raw.replace("loop:", "", 1).strip()

            try:
                inst = Parser.parse_line(clean_raw, i*4, i, labels_map=labels_map)
                if inst:
                    inst.raw_line = raw # Preserva a label 'loop:' para a renderização visual
                    stream_objs.append(inst)
            except SyntaxErrorTomasulo as e:
                self.error_html.value = f"<b style='color:red;'>❌ Erro de Sintaxe: {str(e)} na instrução '{raw}'</b>"
                return

        if not stream_objs:
            self.error_html.value = "<b style='color:orange;'>⚠️ Código vazio.</b>"; return

        try:
            # Instanciação das engines
            self.core_a, self.is_tomasulo_a = self.create_core_helper(self.panel_a, copy.deepcopy(stream_objs), initial_regs, initial_mem)
            self.core_b, self.is_tomasulo_b = self.create_core_helper(self.panel_b, copy.deepcopy(stream_objs), initial_regs, initial_mem)

            self.build_execution_ui()
            self.render_both_states()

        except Exception as e:
            self.error_html.value = f"<b style='color:red;'>❌ Ocorreu um erro ao inicializar os núcleos: {str(e)}</b>"

    def build_execution_ui(self):
        with self.out:
            self.btn_prev = widgets.Button(description="<< Anterior", button_style='warning', layout=widgets.Layout(width='150px'))
            self.btn_next = widgets.Button(description="Próximo >>", button_style='success', layout=widgets.Layout(width='150px'))
            self.btn_next.on_click(self.on_next)
            self.btn_prev.on_click(self.on_prev)

            self.out_a = widgets.Output(layout=widgets.Layout(width='50%', height='600px', overflow='auto', border='1px solid rgba(128,128,128,0.2)', padding='5px'))
            self.out_b = widgets.Output(layout=widgets.Layout(width='50%', height='600px', overflow='auto', border='1px solid rgba(128,128,128,0.2)', padding='5px'))

            display(widgets.HBox([self.btn_prev, self.btn_next], layout=widgets.Layout(margin='10px 0px', justify_content='center')))
            display(widgets.HBox([self.out_a, self.out_b], layout=widgets.Layout(width='100%')))

    def on_next(self, b):
        self.core_a.step()
        self.core_b.step()
        self.render_both_states()

    def on_prev(self, b):
        acted = False
        if self.core_a.history:
            self.core_a.__dict__.update(self.core_a.history.pop()); acted = True
        if self.core_b.history:
            self.core_b.__dict__.update(self.core_b.history.pop()); acted = True
        if acted: self.render_both_states()

    def render_both_states(self):
        self.render_core_ui(self.core_a, self.is_tomasulo_a, self.out_a, "MÁQUINA A")
        self.render_core_ui(self.core_b, self.is_tomasulo_b, self.out_b, "MÁQUINA B")

    def render_core_ui(self, core, is_tomasulo, output_widget, title_label):
        with output_widget:
            clear_output(wait=True)

            completed_insts = sum(
                1 for inst in core.instructions
                if not inst.flushed and (inst.cycle_write is not None or (inst.op == 'bne' and inst.cycle_issue is not None))
            )
            cpi_str = f"{core.cycle / completed_insts:.2f}" if completed_insts > 0 else "0.00"

            html = """
            <style>
                .t-table { width: 100%; border-collapse: collapse; font-family: Courier, monospace; font-size: 11px; margin-bottom: 10px; color: inherit; }
                .t-table th, .t-table td { border: 1px solid rgba(128, 128, 128, 0.4); padding: 2px 3px; text-align: center; }
                .t-table th { background-color: rgba(128, 128, 128, 0.2); font-weight: bold; }
                .t-iter { background-color: rgba(128, 128, 128, 0.1); }
                .hl-change { animation: blink 1.5s ease-out; font-weight: bold; background-color: rgba(245, 158, 11, 0.3); }
                @keyframes blink { 0% { background-color: rgba(245, 158, 11, 0.7); } 100% { background-color: transparent; } }
                .panel-section { display: flex; flex-direction: column; gap: 8px; }
            </style>
            """
            m_str = "Tomasulo (OoO)" if is_tomasulo else "Simples (In-Order)"
            html += f"<div style='background: rgba(128, 128, 128, 0.1); padding: 6px; border-radius: 4px; margin-bottom: 8px; font-size: 12px; border-left: 4px solid #f59e0b; color: inherit;'>"
            html += f"<b>{title_label}</b> | {m_str}<br><b>Ciclo Atual:</b> {core.cycle} | <b>CPI:</b> {cpi_str}</div>"

            insts_by_id = {}
            for inst in core.instructions:
                if inst.id not in insts_by_id: insts_by_id[inst.id] = []
                insts_by_id[inst.id].append(inst)

            max_iters = max((len(lst) for lst in insts_by_id.values()), default=0)
            html += "<table class='t-table'><tr><th rowspan='2'>Instrução</th>"
            for it in range(1, max_iters + 1): html += f"<th colspan='4' class='t-iter'>Iteração {it}</th>"
            html += "</tr><tr>"
            for it in range(max_iters): html += "<th>F</th><th>I</th><th>E</th><th>W</th>"
            html += "</tr>"

            for i in sorted(insts_by_id.keys()):
                lst = insts_by_id[i]
                line_text = lst[0].raw_line if not lst[-1].flushed else f"<s style='opacity: 0.5;'>{lst[0].raw_line}</s>"
                html += f"<tr><td style='text-align:left; font-weight:bold;'>{line_text}</td>"

                for it in range(max_iters):
                    if it < len(lst):
                        curr = lst[it]
                        f_txt = ""
                        if curr.cycle_fetch_start is not None:
                            f_txt = f"{curr.cycle_fetch_start}-..." if curr.cycle_fetch is None else (f"{curr.cycle_fetch}" if curr.cycle_fetch_start == curr.cycle_fetch else f"{curr.cycle_fetch_start}-{curr.cycle_fetch}")

                        i_txt = ""
                        if curr.cycle_issue is not None:
                            d_s = curr.cycle_fetch + 1
                            i_txt = f"{d_s}-{curr.cycle_issue}" if d_s < curr.cycle_issue else str(curr.cycle_issue)
                        elif curr.cycle_fetch is not None:
                            if core.cycle >= curr.cycle_fetch + 1: i_txt = f"{curr.cycle_fetch + 1}-..."

                        e_txt, w_txt = "", ""
                        if curr.op == 'bne':
                            if curr.cycle_issue is not None: e_txt, w_txt = "-", "-"
                        else:
                            if curr.cycle_issue is not None:
                                wait_s = curr.cycle_issue + 1
                                if curr.cycle_exec_start is None:
                                    if core.cycle >= wait_s: e_txt = f"{wait_s}-..."
                                else:
                                    s, e = curr.cycle_exec_start, curr.cycle_exec_end
                                    e_txt = f"{wait_s}-{s}-{e}" if wait_s < s else (f"{s}" if s == e else f"{s}-{e}")
                                    if curr.cycle_write is not None:
                                        if curr.cycle_write > e + 1: e_txt += f"-{curr.cycle_write - 1}"
                                    elif core.cycle > e: e_txt += f"-..."
                                w_txt = str(curr.cycle_write) if curr.cycle_write else ""
                        html += f"<td>{f_txt}</td><td>{i_txt}</td><td>{e_txt}</td><td>{w_txt}</td>"
                    else: html += "<td></td><td></td><td></td><td></td>"
                html += "</tr>"
            html += "</table>"

            prev = core.history[-1] if core.history else None
            def td(curr, d_prev, k, default=""):
                c_val = curr.get(k, default)
                if d_prev and c_val != d_prev.get(k, default): return f"<td class='hl-change'>{c_val}</td>"
                return f"<td>{c_val}</td>"

            html += "<div class='panel-section'>"

            if is_tomasulo:
                html += "<div><table class='t-table'><tr><th>Estação</th><th>Busy</th><th>Op</th><th>Vj</th><th>Vk</th><th>Qj</th><th>Qk</th></tr>"
                for rs_list in core.stations.values():
                    for rs in rs_list:
                        op = rs.inst.raw_line.split()[-1] if rs.inst else ""
                        html += f"<tr><td>{rs.name}</td><td>{'S' if rs.busy else ''}</td><td>{op}</td>"
                        html += f"<td>{rs.Vj if rs.Vj is not None else ''}</td><td>{rs.Vk if rs.Vk is not None else ''}</td>"
                        html += f"<td>{rs.Qj or ''}</td><td>{rs.Qk or ''}</td></tr>"
                html += "</table></div>"

            def render_banco(prefix, regs, status_dict, prev_regs):
                lbl = "Qi (RS)" if is_tomasulo else "Bloqueio (I.id)"
                res = f"<div><table class='t-table'><tr><th>{prefix.upper()}</th><th>Valor</th><th>{lbl}</th></tr>"
                for idx in range(1, 21):
                    reg = f"{prefix}{idx}"
                    val = regs[reg]
                    st = status_dict[reg]
                    st_str = st if is_tomasulo else (f"Ocup (I.{st.id})" if st else "")
                    if val != 0 or st is not None or (prev_regs and prev_regs[reg] != val):
                        res += f"<tr><td><b>{reg}</b></td>{td(regs, prev_regs, reg)}<td>{st_str}</td></tr>"
                res += "</table></div>"
                return res

            html += "<div style='display:flex; justify-content:space-between; gap:5px;'>"\
                    f"{render_banco('f', core.regs_f, core.regs_status_f, prev['regs_f'] if prev else None)}"\
                    f"{render_banco('r', core.regs_r, core.regs_status_r, prev['regs_r'] if prev else None)}"\
                    "</div>"

            html += "<div><table class='t-table'><tr><th>Endereço Memória</th><th>Valor Armazenado</th></tr>"
            accessed = sorted(list(core.accessed_memory))
            if not accessed: html += "<tr><td colspan='2' style='color:gray;'><i>Nenhum acesso</i></td></tr>"
            else:
                for addr in accessed:
                    val = core.memory.get(addr, 0.0)
                    pv = prev['memory'].get(addr, 0.0) if prev else val
                    hl = "class='hl-change'" if prev and pv != val else ""
                    html += f"<tr><td>[{addr}]</td><td {hl}>{val}</td></tr>"
            html += "</table></div></div>"
            display(HTML(html))

### *Parsing Assembly* $\rightarrow$ *Python*

In [44]:
# TABELA DE INSTRUCOES
INSTR_TABLE = {
    "mulf":  ("r_type", "*"),
    "multf": ("r_type", "*"),
    "mul":   ("r_type", "*"),
    "mult":  ("r_type", "*"), # Todos os muls viram mulf
    "addf":  ("r_type", "+"),
    "add":   ("r_type", "+"),
    "sub":   ("r_type", "-"),
    "addi":  ("i_type", "+"),
    "subi":  ("i_type", "-"),
    "ld":    ("load", None),
    "sd":    ("store", None),
    "bne":   ("branch", "!="),
    "beq":   ("branch", "=="),
    "j":     ("jump", None),
    "mov":   ("mov", None)
}



MEM_RE = re.compile(r"^(-?\d+)\((\w+)\)$")

def _reg_name(token):
    m = re.match(r"^([fr])(\d+)$", token)
    if m:
        bank, idx = m.groups()
        return f"{bank}[{idx}]"
    return token

def parse_line(raw_line, line_num=None):
    line = raw_line.split("#")[0].strip()
    if not line:
        return None

    # Aceita instruções, registradores e labels em UPPERCASE ou lowercase
    line = line.lower()

    label = None
    # Separação robusta de label (funciona com ou sem espaço após o ':')
    if ":" in line:
        parts = line.split(":", 1)
        possible_label = parts[0].strip()
        # Garante que não há espaços internos no nome da label
        if " " not in possible_label and possible_label:
            label = possible_label
            line = parts[1].strip()
            if not line:
                return {"type": "label", "label": label, "raw": raw_line.strip(),
                        "python": f"# label: {label}"}

    m = re.match(r"^(\S+)\s+(.*)$", line)
    if not m:
        raise ValueError(f"Linha {line_num}: não consegui interpretar '{raw_line.strip()}'")
    opcode, rest = m.groups()

    # 💥 CORREÇÃO: Extrai os operandos PRIMEIRO para que fiquem disponíveis na checagem abaixo
    operands = [op.strip() for op in rest.split(",")]

    # Normalização de opcodes para a identidade padrão da máquina (Tomasulo friendly)
    if opcode in ["mulf", "multf", "mul", "mult"]:
        opcode = "mulf"
    elif opcode == "add" and any('f' in op for op in operands):
        opcode = "addf"

    if opcode not in INSTR_TABLE:
        raise ValueError(f"instrução desconhecida '{opcode}'")

    category, op = INSTR_TABLE[opcode]
    result = {"opcode": opcode, "category": category, "op": op, "label": label,
              "raw": raw_line.strip(), "operands": operands}

    expected_operands = {
        "r_type": 3, "i_type": 3, "load": 2, "store": 2, "branch": 3, "jump": 1, "mov": 2
    }
    if len(operands) != expected_operands[category]:
        raise ValueError(f"'{opcode}' espera {expected_operands[category]} operando(s), recebeu {len(operands)}")

    if category == "r_type":
        dest, src1, src2 = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src1)} {op} {_reg_name(src2)}"
    elif category == "i_type":
        dest, src, imm = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)} {op} {imm}"
    elif category == "load":
        dest, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"{_reg_name(dest)} = mem[{_reg_name(base)} + {offset}]"
    elif category == "store":
        src, mem = operands
        mm = MEM_RE.match(mem)
        if not mm: raise ValueError(f"endereço de memória inválido '{mem}'")
        offset, base = mm.groups()
        result["python"] = f"mem[{_reg_name(base)} + {offset}] = {_reg_name(src)}"
    elif category == "mov":
        dest, src = operands
        result["python"] = f"{_reg_name(dest)} = {_reg_name(src)}"
    elif category == "branch":
        src1, src2, target = operands
        result["python"] = f"if {_reg_name(src1)} {op} {_reg_name(src2)}: goto('{target}')"
    elif category == "jump":
        (target,) = operands
        result["python"] = f"goto('{target}')"

    if label:
        result["python"] = f"# {label}:\n" + result["python"]
    return result

def extract_dest_reg(text):
    """Extrai o registrador de destino de uma instrução se ele for de ponto flutuante (fX)"""
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    # Ignora instruções que não escrevem em registrador de destino flutuante
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt', 'j']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def parse_program(text):
    out = []
    for i, raw_line in enumerate(text.split("\n"), 1):
        parsed = parse_line(raw_line, line_num=i)
        if parsed:
            out.append(parsed)
    return out

---

## Ferramenta interativa

In [45]:
#@title Editor de Instruções
custom_css = widgets.HTML("""
<style>
.header-card {
    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);
    padding: 18px 24px; border-radius: 14px 14px 0 0; color: white; font-family: 'Segoe UI', sans-serif;
}
.header-card h3 { margin: 0; font-size: 20px; font-weight: 600; }
.header-card p { margin: 4px 0 0 0; font-size: 13px; opacity: 0.85; }
.tool-card { border: 1px solid #dcdde1; border-radius: 14px; padding: 0 0 18px 0; background: #fafbfc; box-shadow: 0 4px 16px rgba(0,0,0,0.08); margin-bottom: 16px; }
.inner-content { padding: 18px 24px 0 24px; }
.widget-textarea textarea { border-radius: 10px !important; border: 1.5px solid #a8dadc !important; font-family: 'Consolas','Courier New',monospace !important; font-size: 20px !important; line-height: 1.5 !important; padding: 12px !important; }
.widget-button { border-radius: 8px !important; font-weight: 600 !important; font-size: 13px !important; height: 38px !important; }
.output-box { background: white; border-radius: 10px; border: 1px solid #e0e0e0; padding: 8px 14px; font-family: 'Consolas','Courier New',monospace; font-size: 15px; color: #1d1d1d !important; }
.output-box pre { color: #1d1d1d !important; background: transparent !important; }
.output-box * { color: #1d1d1d !important; }
.config-panel { background: #eef2f7; border: 1px solid #dcdde1; border-radius: 10px; padding: 14px 20px; margin: 10px 0; font-family: 'Segoe UI', sans-serif; }
</style>
""")

# ---- estado do programa e da máquina fixa (compartilhados entre as células) ----
program_instructions = []
machine_config = MachineConfig(mem_size=200, word_size=4)

# ---------------- Widgets de entrada do programa ----------------
input_area = widgets.Textarea(
    value=(
        'loop:ld f3,0(r1)\n'
        'multf f1,f2,f3\n'
        'ld f4,4(r1)\n'
        'mulf f5,f1,f4\n'
        'mult f4,f4,f2\n'
        'sd f5,0(r1)\n'
        'addf f5,f5,f4\n'
        'sd f5,4(r1)\n'
        'addi r1,r1,8\n'
        'bne r1,r2,loop'
    ),
    description='',
    layout=widgets.Layout(width='95%', height='320px', margin='0 0 12px 0')
)

add_button = widgets.Button(description='Adicionar ao programa', icon='plus', button_style='success', layout=widgets.Layout(width='220px'))
clear_button = widgets.Button(description='Limpar programa', icon='trash', button_style='danger', layout=widgets.Layout(width='180px'))
graph_button = widgets.Button(description='Gerar grafo de dependências', icon='project-diagram', button_style='info', layout=widgets.Layout(width='260px'))

output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0', max_height='300px', overflow='auto'))
graph_output = widgets.Output(layout=widgets.Layout(margin='10px 0 0 0'))

def on_add_clicked(b):
    with output:
        clear_output()
        try:
            novas = parse_program(input_area.value)
            program_instructions.extend(novas)
            print(f"✅ {len(novas)} instrução(ões) adicionada(s). Total no programa: {len(program_instructions)}\n")
            for i, instr in enumerate(program_instructions):
                print(f"[{i}] {instr['raw']}  ->  {instr.get('python', '<<< SEM TRADUÇÃO >>>')}")
        except ValueError as e:
            print(f"❌ Erro de parsing: {e}")

def on_clear_clicked(b):
    global program_instructions
    program_instructions = []
    input_area.value = ''
    with output:
        clear_output(); print("🗑️ Programa limpo.")
    with graph_output:
        clear_output()

def build_dependency_graph(instructions):
    G = nx.DiGraph()
    for i, instr in enumerate(instructions):
        if instr.get('type') == 'label':
            continue

        opcode = instr['opcode']
        if opcode in ["mulf", "multf", "mul", "mult"]:
            opcode = "mult"
        elif opcode in ["addf", "add"]:
            opcode = "add"

        display_text = f"{opcode} {', '.join(instr['operands'])}"
        G.add_node(i, text=display_text)

    last_write = {}
    edges = defaultdict(set)

    for j, instr in enumerate(instructions):
        if instr.get('type') == 'label' or instr['category'] in ('branch', 'jump'):
            continue

        opcode = instr['opcode']
        operands = instr['operands']
        find_f = lambda t: set(re.findall(r'f\d+', t.lower()))

        writes_j, reads_j = set(), set()

        if opcode in ('mult', 'mul', 'mulf', 'multf'):
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        # 🔧 FIX: Incluído addi, sub e subi no mapeamento
        elif opcode in ('add', 'addf', 'addi', 'sub', 'subi'):
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode == 'ld':
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
        elif opcode == 'sd':
            reads_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])
        elif opcode == 'mov':
            writes_j |= find_f(operands[0])
            if len(operands) > 1: reads_j |= find_f(operands[1])

        for r in reads_j:
            if r in last_write:
                edges[(last_write[r], j)].add(r)

        for r in writes_j:
            last_write[r] = j

    for (u, v), regs in edges.items():
        G.add_edge(u, v, regs=regs)
    return G

def compute_layered_positions(G):
    level = {}
    for node in nx.topological_sort(G):
        preds = list(G.predecessors(node))
        level[node] = 0 if not preds else max(level[p] for p in preds) + 1
    return level

def on_graph_clicked(b):
    with graph_output:
        clear_output()
        if not program_instructions:
            print("⚠️ Nenhuma instrução no programa ainda. Adicione instruções primeiro.")
            return
        G = build_dependency_graph(program_instructions)
        no_dep = [n for n in G.nodes() if G.degree(n) == 0]
        G.remove_nodes_from(no_dep)
        if G.number_of_nodes() == 0:
            print("ℹ️ Nenhuma instrução com dependência de registradores 'f' para exibir.")
            return

        level = compute_layered_positions(G)
        levels = defaultdict(list)
        for node, lvl in level.items():
            levels[lvl].append(node)

        dot = graphviz.Digraph(comment='Grafo de Dependencia Puro RAW', format='svg')
        dot.attr(rankdir='TB', splines='true', nodesep='1.1', ranksep='0.7')

        dot.attr('node', fontname='monospace', shape='ellipse', style='filled',
                 fillcolor='#f1faee', color='#1d3557', penwidth='2.2', fontcolor='#1d3557', fontsize='12')
        dot.attr('edge', fontname='sans-serif', color='#e63946', penwidth='1.8', fontsize='11', fontcolor='#e63946', weight='1.2')

        max_level = max(level.values())

        for lvl in range(max_level + 1):
            dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none',
                     color='none', fontcolor='#2d3436', fontsize='13', fontweight='bold')
            if lvl > 0:
                dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

        for lvl in sorted(levels.keys()):
            with dot.subgraph() as s:
                s.attr(rank='same')
                s.node(f'L_{lvl}')
                for node in levels[lvl]:
                    s.node(str(node), G.nodes[node]['text'])

        for u, v, data in G.edges(data=True):
            reg_label = ", ".join(sorted(data['regs']))
            dot.edge(str(u), str(v), label=f" {reg_label} ")

        display(dot)

add_button.on_click(on_add_clicked)
clear_button.on_click(on_clear_clicked)
graph_button.on_click(on_graph_clicked)

output.add_class('output-box')
graph_output.add_class('output-box')

header = widgets.HTML("""
<div class="header-card">
    <h3>🔧 Editor de Instruções — Programa + Configuração da Máquina</h3>
    <p>Digite instruções Assembly, adicione ao programa e gere o grafo de dependências.</p>
</div>
""")

config_panel = widgets.HTML(f"""
<div class="config-panel">
    <span style="color: #1d3557; font-weight: bold; font-size: 14px;">⚙️ Configuração Fixa da Máquina:</span><br/>
    <span style="font-size: 13px; color: #2d3436;">
        • <b>Tamanho da Memória:</b> {machine_config.mem_size} posições |
        • <b>Word Size:</b> {machine_config.word_size} bytes<br/>
        • <b>Estado Inicial dos Registradores:</b> <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">r1=1, r2=2, ..., r31=31</code> e <code style="background: #ffffff; padding: 2px 4px; border-radius: 4px;">f1=1.0, f2=2.0, ..., f31=31.0</code>
    </span>
</div>
""")

body = widgets.VBox([
    input_area,
    widgets.HBox([add_button, clear_button, graph_button], layout=widgets.Layout(gap='10px')),
    config_panel,
    output,
    graph_output
], layout=widgets.Layout())
body.add_class('inner-content')

card = widgets.VBox([header, body])
card.add_class('tool-card')

display(custom_css, card)

HTML(value="\n<style>\n.header-card {\n    background: linear-gradient(135deg, #1d3557 0%, #457b9d 100%);\n   …

### Gera grafo de soft. pipeline e compara com original

In [55]:
#@title Usuário propõe grafo e possui gabarito { display-mode: "form" }

# =========================================================================
# 🧬 ESTILOS CSS - RESPONSIVIDADE E CORREÇÃO DAS ABAS (DARK MODE)
# =========================================================================
css_unificado = widgets.HTML("""
    <style>
    .no-pointer-plots canvas, .no-pointer-plots .jupyter-matplotlib { pointer-events: none !important; }
    /* Mantém os grafos de ambas as abas 100% responsivos */
    .responsive-graph svg, .responsive-anim-graph svg { width: 100% !important; height: auto !important; max-width: 100% !important; }

    /* 🚀 CORREÇÃO DEFINITIVA DAS ABAS DO IPYWIDGETS NO COLAB */
    .lm-TabBar-tab, .p-TabBar-tab {
        width: auto !important;
        min-width: 220px !important;
        padding: 0 20px !important;
        overflow: visible !important;
    }
    .lm-TabBar-tabLabel, .p-TabBar-tabLabel {
        color: #ffffff !important;
        font-weight: 600 !important;
        font-size: 13px !important;
        overflow: visible !important;
        text-overflow: clip !important;
    }
    .lm-TabBar-tab:not(.lm-mod-current), .p-TabBar-tab:not(.p-mod-current) {
        background-color: #334155 !important;
        opacity: 0.7;
    }
    .lm-TabBar-tab.lm-mod-current, .p-TabBar-tab.p-mod-current {
        background-color: #1e293b !important;
        border-bottom: 2px solid #38bdf8 !important;
        opacity: 1;
    }
    </style>
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">🛠️ Central de Grafos: Pipeline de Software</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha o seu grafo no Canvas ou navegue pelo Gabarito Evolutivo gerado automaticamente.</p>
    </div>
""")

# =========================================================================
# 📦 SUPORTE À EXPORTAÇÃO INDIVIDUAL DE SVGs
# =========================================================================
import os
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

EXPORT_DIR = '/content/svg_exports' if IN_COLAB else './svg_exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

def _save_and_maybe_download(dot, filename_no_ext):
    """Renderiza o objeto graphviz.Digraph em .svg no disco e dispara download no Colab."""
    rendered_path = dot.render(filename=filename_no_ext, directory=EXPORT_DIR, format='svg', cleanup=True)
    if IN_COLAB:
        colab_files.download(rendered_path)
    return rendered_path

def make_export_button(get_dot_fn, filename, feedback_output, label='Exportar SVG'):
    """
    Cria um botão de exportação que só resolve o objeto Digraph no momento do clique
    (via get_dot_fn), evitando capturar um grafo desatualizado no closure.
    """
    btn = widgets.Button(
        description=label, icon='download', button_style='',
        layout=widgets.Layout(width='150px', height='28px', margin='0 0 8px 0')
    )
    def _handler(b):
        dot_obj = get_dot_fn()
        if dot_obj is None:
            with feedback_output:
                print(f"⚠️ Nada para exportar ainda em '{filename}'.")
            return
        try:
            path = _save_and_maybe_download(dot_obj, filename)
            with feedback_output:
                print(f"✅ Exportado: {path}")
        except Exception as e:
            with feedback_output:
                print(f"❌ ERRO AO EXPORTAR '{filename}': {e}")
    btn.on_click(_handler)
    return btn

# =========================================================================
# ✏️ CONFIGURAÇÃO DE WIDGETS - ABA 1: CANVAS DO USUÁRIO
# =========================================================================
txt_node_instruction = widgets.Text(value='mov f4, f1', placeholder='Ex: mult f2, f1, f1', description='Instrução:', layout=widgets.Layout(width='280px'))
slider_node_level = widgets.IntSlider(value=0, min=0, max=8, step=1, description='Nível (N):', layout=widgets.Layout(width='240px'))
btn_add_canvas_node = widgets.Button(description='Adicionar Nó', icon='plus', button_style='success', layout=widgets.Layout(width='140px', height='34px'))

txt_graph_code = widgets.Textarea(
    value='# Formato esperado: [nivel] inst\n[0] ld f3, 0(r1)\n[0] ld f4, 4(r1)\n[1] mult f1, f2, f3\n[1] mov f6, f4\n[1] mult f7, f4, f2\n[2] mult f5, f1, f6\n[2] mov f8, f7\n[3] sd f5, 0(r1)\n[3] add f9, f5, f8\n[4] sd f9, 4(r1)',
    placeholder='Digite o grafo por extenso...',
    description='Script Grafo:',
    layout=widgets.Layout(width='675px', height='125px', margin='5px 0 10px 0')
)
btn_load_graph_code = widgets.Button(description='Carregar Grafo por Escrito', icon='code', button_style='info', layout=widgets.Layout(width='230px', height='34px', margin='40px 0 0 0'))

drop_remove_node = widgets.Dropdown(options=[('Nenhum nó disponível', -1)], description='Selecionar Nó:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'), disabled=True)
btn_remove_canvas_node = widgets.Button(description='Remover Selecionado', icon='trash', button_style='warning', layout=widgets.Layout(width='180px', height='34px'))
btn_clear_canvas = widgets.Button(description='Limpar Canvas', icon='trash-restore', button_style='danger', layout=widgets.Layout(width='140px', height='34px'))
btn_validate_canvas_graph = widgets.Button(description='Validar Grafo Proposto', icon='shield-check', button_style='primary', layout=widgets.Layout(width='220px', height='34px'))

# ✨ AQUI ESTÁ A CORREÇÃO: Criação das duas saídas de tela
canvas_output_plot = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))
canvas_validation_output = widgets.Output(layout=widgets.Layout(margin='10px 0 0 0'))
user_canvas_nodes = []

# =========================================================================
# 📖 CONFIGURAÇÃO DE WIDGETS - ABA 2: GABARITO PASSO A PASSO
# =========================================================================
btn_graph_prev = widgets.Button(description='Passo Anterior', icon='arrow-left', button_style='warning', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_next = widgets.Button(description='Próximo Passo', icon='arrow-right', button_style='success', layout=widgets.Layout(width='160px', height='40px'))
btn_graph_final = widgets.Button(description='Ir para o Final', icon='fast-forward', button_style='info', layout=widgets.Layout(width='160px', height='40px'))

lbl_graph_step = widgets.Label(value='Passo 0 de 0', layout=widgets.Layout(margin='8px 0 0 15px'))
lbl_graph_step.style.text_color = '#f1f5f9'
pipe_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

current_graph_idx = 0
pipeline_cache = {}

# =========================================================================
# 🧮 FUNÇÕES DE SUPORTE DO CANVAS (ABA 1)
# =========================================================================
def update_remove_dropdown():
    if not user_canvas_nodes:
        drop_remove_node.options = [('Nenhum nó disponível', -1)]; drop_remove_node.disabled = True
    else:
        drop_remove_node.options = [(f"[{idx}] {item['text']} (Nível {item['level']})", idx) for idx, item in enumerate(user_canvas_nodes)]
        drop_remove_node.disabled = False

def build_user_proposed_graph(nodes_list):
    G = nx.DiGraph()
    for idx, item in enumerate(nodes_list):
        parsed = parse_line(item['text'], 0)
        opcode = parsed['opcode'] if parsed else 'mov'
        if opcode in ["mulf", "multf", "mul", "mult"]: opcode = "mult"
        elif opcode in ["addf", "add"]: opcode = "add"
        cleaned_text = f"{opcode} {', '.join(parsed['operands'])}" if parsed else item['text']
        G.add_node(idx, text=cleaned_text, level=item['level'])

    last_write = {}
    edges = defaultdict(set)
    for j, item in enumerate(nodes_list):
        parsed = parse_line(item['text'], 0)
        if not parsed: continue
        opcode = parsed['opcode']
        if opcode in ["mulf", "multf", "mul", "mult"]: opcode = "mult"
        elif opcode in ["addf", "add"]: opcode = "add"
        operands = parsed['operands']
        find_f = lambda t: set(re.findall(r'f\d+', t.lower()))

        writes_j, reads_j = set(), set()
        # 🔧 FIX: Incluído addi, sub e subi no mapeamento compacto
        if opcode in ('mult', 'add', 'sub', 'addi', 'subi'):
            writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode == 'ld': writes_j |= find_f(operands[0])
        elif opcode == 'sd': reads_j |= find_f(operands[0])
        elif opcode == 'mov': writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])

        for r in reads_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
        for r in writes_j:
            last_write[r] = j

    for (u, v), regs in edges.items(): G.add_edge(u, v, regs=regs)
    return G

    last_write = {}
    edges = defaultdict(set)
    for j, item in enumerate(nodes_list):
        parsed = parse_line(item['text'], 0)
        if not parsed: continue
        opcode = parsed['opcode']
        if opcode in ["mulf", "multf", "mul", "mult"]: opcode = "mult"
        elif opcode in ["addf", "add"]: opcode = "add"
        operands = parsed['operands']
        find_f = lambda t: set(re.findall(r'f\d+', t.lower()))

        writes_j, reads_j = set(), set()
        if opcode in ('mult', 'add', 'sub'):
            writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])
            if len(operands) > 2: reads_j |= find_f(operands[2])
        elif opcode == 'ld': writes_j |= find_f(operands[0])
        elif opcode == 'sd': reads_j |= find_f(operands[0])
        elif opcode == 'mov': writes_j |= find_f(operands[0]); reads_j |= find_f(operands[1])

        for r in reads_j:
            if r in last_write: edges[(last_write[r], j)].add(r)
        for r in writes_j:
            last_write[r] = j

    for (u, v), regs in edges.items(): G.add_edge(u, v, regs=regs)
    return G

def make_html_label_local(text, dest_changed=False):
    text = text.lower().strip()
    parts = text.split(None, 1)
    if len(parts) == 2:
        opcode, rest = parts; subparts = rest.split(',', 1)
        if len(subparts) == 2:
            dest, remaining = subparts
            color = "red" if dest_changed else "#1d3557"
            return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{opcode} </b></font></td><td><font color="{color}"><b>{dest.strip()}</b></font></td><td><font color="#1d3557"><b>, {remaining.strip()}</b></font></td></tr></table>>'
    return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{text}</b></font></td></tr></table>>'

def generate_graphviz_panel(G, level_dict, node_text_map, dest_changed_map=None):
    dot = graphviz.Digraph(format='svg')
    dot.attr(rankdir='TB', splines='true', nodesep='1.0', ranksep='0.6')
    dot.attr('node', fontname='monospace', shape='ellipse', style='filled', fillcolor='white', color='#1e293b', penwidth='2.0', fontcolor='#1e293b', fontsize='12')
    dot.attr('edge', fontname='sans-serif', color='#475569', penwidth='1.8', fontsize='11', fontcolor='#475569')

    if not level_dict: return dot
    max_level = max(level_dict.values())
    for lvl in range(max_level + 1):
        dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none', color='none', fontcolor='#475569', fontsize='13', fontweight='bold')
        if lvl > 0: dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

    levels_to_nodes = defaultdict(list)
    for n, lvl in level_dict.items(): levels_to_nodes[lvl].append(n)

    for lvl in sorted(levels_to_nodes.keys()):
        with dot.subgraph() as s:
            s.attr(rank='same'); s.node(f'L_{lvl}')
            for node in levels_to_nodes[lvl]:
                is_changed = dest_changed_map[node] if dest_changed_map else False
                lbl = make_html_label_local(node_text_map[node], is_changed)
                s.node(str(node), lbl)

    for u, v, data in G.edges(data=True):
        dot.edge(str(u), str(v), label=f" {', '.join(sorted(data['regs']))} ")
    return dot

def render_canvas_split_view():
    with canvas_output_plot:
        clear_output(wait=True)
        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)

        if G_orig.number_of_nodes() == 0:
            print("⚠️ Aguardando carregamento de instruções válidas na Célula 2.")
            return

        level_orig = compute_layered_positions(G_orig)
        node_text_map_orig = {n: G_orig.nodes[n]['text'] for n in G_orig.nodes()}
        dot_orig = generate_graphviz_panel(G_orig, level_orig, node_text_map_orig)

        original_written = ISASimulator.collect_written_regs(program_instructions)
        G_user = build_user_proposed_graph(user_canvas_nodes)
        level_user = {idx: item['level'] for idx, item in enumerate(user_canvas_nodes)}
        node_text_map_user = {idx: item['text'] for idx, item in enumerate(user_canvas_nodes)}

        def is_renamed(text):
            parts = text.split(None, 1)
            if len(parts) < 2: return False
            opcode, rest = parts; dest = rest.split(',')[0].strip()
            return opcode == 'mov' or dest not in original_written

        dest_changed_user = {idx: is_renamed(item['text']) for idx, item in enumerate(user_canvas_nodes)}
        dot_user = generate_graphviz_panel(G_user, level_user, node_text_map_user, dest_changed_user) if user_canvas_nodes else None

        out_left, out_right = widgets.Output(), widgets.Output()
        out_left.add_class('responsive-graph')
        out_right.add_class('responsive-graph')

        with out_left: display(dot_orig)
        with out_right:
            if user_canvas_nodes: display(dot_user)
            else: print("\n\n\n\n[ Canvas em Branco ]")

        title_style = 'color: #ffffff; text-align: center; font-weight: bold; font-family: sans-serif; font-size: 18px; margin: 0 0 15px 0; padding: 0;'

        # 📤 Botões de exportação próximos a cada respectivo grafo
        btn_export_orig = make_export_button(
            get_dot_fn=lambda: dot_orig, filename='grafo_original',
            feedback_output=canvas_output_plot, label='Exportar Original'
        )
        btn_export_user = make_export_button(
            get_dot_fn=lambda: dot_user, filename='grafo_usuario',
            feedback_output=canvas_output_plot, label='Exportar Proposto'
        )
        if dot_user is None:
            btn_export_user.disabled = True

        col_left = widgets.VBox(
            [widgets.HTML(f'<h3 style="{title_style}">1. Grafo Original (Referência)</h3>'),
             widgets.HBox([btn_export_orig], layout=widgets.Layout(justify_content='center')),
             out_left],
            layout=widgets.Layout(width='50%')
        )
        col_right = widgets.VBox(
            [widgets.HTML(f'<h3 style="{title_style}">2. Seu Grafo Proposto</h3>'),
             widgets.HBox([btn_export_user], layout=widgets.Layout(justify_content='center')),
             out_right],
            layout=widgets.Layout(width='50%')
        )
        split_panel = widgets.HBox([col_left, col_right], layout=widgets.Layout(width='100%', align_items='flex-start'))
        display(split_panel)

def on_validate_canvas_clicked(b):
    # ✨ FIX: Usa a saída dedicada para feedback visual imediato
    with canvas_validation_output:
        clear_output(wait=True)
        print("⏳ Validando o pipeline, aguarde...")

    if not user_canvas_nodes:
        with canvas_validation_output:
            clear_output()
            print("\n❌ ERRO DE VALIDAÇÃO: O canvas está completamente vazio!")
            print("Insira instruções nas camadas ou use o 'Script Grafo' antes de submeter para homologação.\n")
        return

    validator = PipelineValidator(program_instructions, machine_config)
    result = validator.validate(user_canvas_nodes)

    with canvas_validation_output:
        clear_output()
        if result.get("error"):
            print(f"❌ {result['error']}")
            return

        n_show = min(30, machine_config.mem_size)
        if result["success"]:
            print("\n✅ VEREDICTO: SEU GRAFO DE PIPELINE ESTÁ CORRETO!")
            print("A distribuição de níveis e as dependências de registradores geraram equivalência semântica perfeita na memória.\n")
        else:
            print("\n❌ VEREDICTO: GRAFO INCORRETO (Hazard ou Renomeação Inválida)")

        print(f"    Gabarito Esperado : {result['mem_gabarito'][:n_show]}")
        print(f"    Sua Saída         : {result['mem_pipe'][:n_show]}")

def on_add_node_clicked(b):
    inst_text = txt_node_instruction.value.strip()
    if inst_text:
        try:
            parsed_result = parse_line(inst_text.lower(), line_num=len(user_canvas_nodes) + 1)
            if parsed_result:
                user_canvas_nodes.append({'text': inst_text.lower(), 'level': slider_node_level.value})
                update_remove_dropdown(); render_canvas_split_view()
        except ValueError as e:
            with canvas_output_plot: print(f"❌ REJEITADO PELO PARSER: {e}")

def on_load_graph_code_clicked(b):
    global user_canvas_nodes
    code_block = txt_graph_code.value.strip()
    if not code_block: return
    parsed_nodes = []
    try:
        for idx, line in enumerate(code_block.split('\n'), 1):
            line_clean = line.strip()
            if not line_clean or line_clean.startswith('#'): continue
            match = re.match(r"^\[\s*(\d+)\s*(?:,\s*\d+\s*)?\]\s*(.+)$", line_clean)
            if not match: raise ValueError(f"Linha {idx}: Formato de tags inválido.")
            lvl_str, inst_raw = match.groups(); lvl = int(lvl_str); inst_raw = inst_raw.strip().lower()
            if parse_line(inst_raw, line_num=idx): parsed_nodes.append({'text': inst_raw, 'level': lvl})
        user_canvas_nodes = parsed_nodes
        update_remove_dropdown(); render_canvas_split_view()
    except ValueError as e:
        with canvas_output_plot: print(f"❌ ERRO AO PARSEAR SCRIPT DO GRAFO: {e}")

def on_remove_node_clicked(b):
    idx_to_remove = drop_remove_node.value
    if idx_to_remove is not None and 0 <= idx_to_remove < len(user_canvas_nodes):
        user_canvas_nodes.pop(idx_to_remove)
        update_remove_dropdown(); render_canvas_split_view()

def on_clear_canvas_clicked(b):
    global user_canvas_nodes; user_canvas_nodes = []
    update_remove_dropdown(); render_canvas_split_view()

btn_add_canvas_node.on_click(on_add_node_clicked)
btn_load_graph_code.on_click(on_load_graph_code_clicked)
btn_remove_canvas_node.on_click(on_remove_node_clicked)
btn_clear_canvas.on_click(on_clear_canvas_clicked)
btn_validate_canvas_graph.on_click(on_validate_canvas_clicked)

# =========================================================================
# 🧮 FUNÇÕES DE SUPORTE DO GABARITO EVOLUTIVO (ABA 2)
# =========================================================================

def extract_dest_reg(text):
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt', 'j']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves_ordered(G_orig, level_orig):
    all_regs = []
    for node in G_orig.nodes(): all_regs.extend(re.findall(r'f\d+', G_orig.nodes[node]['text']))
    reg_indices = [int(r[1:]) for r in all_regs if r.startswith('f')]
    global_next_reg = (max(reg_indices) + 1) if reg_indices else 1

    G_pipe = nx.DiGraph()
    level_pipe, node_text_map, dest_changed_map, nodes_order = {}, {}, {}, []
    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items(): levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes, move_node_counter, seen_destinations = {}, 1000, set()
    prod_consumers = defaultdict(list)

    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']: prod_consumers[(u, r)].append(v)
    for u in G_orig.nodes():
        d = extract_dest_reg(G_orig.nodes[u]['text'])
        if d: reg_name_at_level[(u, d)][level_orig[u]] = d

    for lvl in sorted(levels_to_nodes.keys()):
        for (p, r), consumers in prod_consumers.items():
            L_p = level_orig[p]; L_end = max(level_orig[c] for c in consumers)
            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter; move_node_counter += 1
                    r_prev = reg_name_at_level[(p, r)][lvl - 1]; r_new = f"f{global_next_reg}"; global_next_reg += 1
                    node_text_map[mov_id] = f"mov {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl; dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id); nodes_order.append(mov_id)
                    mov_nodes[(p, r, lvl)] = mov_id; reg_name_at_level[(p, r)][lvl] = r_new
                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r, lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else: reg_name_at_level[(p, r)][lvl] = reg_name_at_level[(p, r)][lvl - 1]

        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text']
            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']: src_mappings[r] = reg_name_at_level[(p_node, r)][lvl - 1]

            orig_dest = extract_dest_reg(orig_text); new_dest = orig_dest; dest_changed = False
            if orig_dest:
                if orig_dest in seen_destinations:
                    new_dest = f"f{global_next_reg}"; global_next_reg += 1; dest_changed = True
                seen_destinations.add(orig_dest)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts; subparts = rest.split(','); is_store = opcode.lower() in ['sd', 'sw', 'sb', 'sh']
                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub)
                        new_subparts.append(updated_sub)
                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else: updated_text = orig_text

            node_text_map[u] = updated_text; level_pipe[u] = lvl; dest_changed_map[u] = dest_changed
            G_pipe.add_node(u); nodes_order.append(u)
            if orig_dest: reg_name_at_level[(u, orig_dest)][lvl] = new_dest
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r, lvl - 1)]
                    G_pipe.add_edge(parent_in_pipe, u, regs={reg_name_at_level[(p_node, r)][lvl - 1]})

    return G_pipe, level_pipe, node_text_map, dest_changed_map, nodes_order

def make_html_label_gabarito(text, dest_changed=False):
    text = text.lower().strip()
    parts = text.split(None, 1)
    if len(parts) == 2:
        opcode, rest = parts; subparts = rest.split(',', 1)
        if len(subparts) == 2:
            dest, remaining = subparts; color = "red" if dest_changed else "#1d3557"
            return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{opcode} </b></font></td><td><font color="{color}"><b>{dest.strip()}</b></font></td><td><font color="#1d3557"><b>, {remaining.strip()}</b></font></td></tr></table>>'
    return f'<<table border="0" cellborder="0" cellspacing="0"><tr><td><font color="#1d3557"><b>{text}</b></font></td></tr></table>>'

def generate_graphviz_incremental(G, level_dict, node_text_map, dest_changed_map, visible_nodes_set, highlight_node=None, is_pipelined=False):
    dot = graphviz.Digraph(format='svg')
    dot.attr(rankdir='TB', splines='true', nodesep='1.0', ranksep='0.6')
    dot.attr('node', fontname='monospace', shape='ellipse', style='filled', fillcolor='white', color='#1e293b', penwidth='2.0', fontcolor='#1e293b', fontsize='12')
    dot.attr('edge', fontname='sans-serif', color='#475569', penwidth='1.8', fontsize='11', fontcolor='#475569')

    if not level_dict or not visible_nodes_set: return dot

    max_level = max(level_dict.values())
    for lvl in range(max_level + 1):
        dot.node(f'L_{lvl}', f'N = {lvl}', shape='plaintext', fillcolor='none', color='none', fontcolor='#475569', fontsize='13', fontweight='bold')
        if lvl > 0: dot.edge(f'L_{lvl-1}', f'L_{lvl}', style='invis')

    levels_to_nodes = defaultdict(list)
    for n in visible_nodes_set: levels_to_nodes[level_dict[n]].append(n)

    for lvl in sorted(levels_to_nodes.keys()):
        if lvl > max_level: break
        with dot.subgraph() as s:
            s.attr(rank='same'); s.node(f'L_{lvl}')
            for node in levels_to_nodes[lvl]:
                is_changed = dest_changed_map[node] if dest_changed_map else False
                lbl = make_html_label_gabarito(node_text_map[node], is_changed)
                if node == highlight_node and is_pipelined:
                    s.node(str(node), lbl, fillcolor='#dcfce7', color='#16a34a', penwidth='3.0')
                else:
                    s.node(str(node), lbl, fillcolor='white', color='#1e293b', penwidth='2.0')

    for u, v, data in G.edges(data=True):
        if u in visible_nodes_set and v in visible_nodes_set:
            reg_label = ", ".join(sorted(data['regs']))
            if v == highlight_node and is_pipelined:
                dot.edge(str(u), str(v), label=f" {reg_label} ", color='#2563eb', penwidth='2.2', fontcolor='#2563eb')
            else:
                dot.edge(str(u), str(v), label=f" {reg_label} ")
    return dot

def update_pipeline_cache():
    global pipeline_cache
    if 'program_instructions' not in globals() or not program_instructions:
        pipeline_cache = {}; return False
    if pipeline_cache.get('raw_program') == program_instructions: return True

    G_orig = build_dependency_graph(program_instructions)
    no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]; G_orig.remove_nodes_from(no_dep)
    if G_orig.number_of_nodes() == 0: pipeline_cache = {}; return False

    level_orig = compute_layered_positions(G_orig)
    G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe, nodes_order = build_pipelined_with_moves_ordered(G_orig, level_orig)

    pipeline_cache = {
        'raw_program': list(program_instructions), 'G_orig': G_orig, 'level_orig': level_orig,
        'G_pipe': G_pipe, 'level_pipe': level_pipe, 'node_text_map_pipe': node_text_map_pipe,
        'dest_changed_map_pipe': dest_changed_map_pipe, 'nodes_order': nodes_order
    }
    return True

def render_current_graph_step():
    with pipe_compare_output:
        clear_output(wait=True)
        if not update_pipeline_cache():
            print("⚠️ Nenhuma instrução com dependência válida encontrada no programa."); return

        c = pipeline_cache; nodes_order = c['nodes_order']
        global current_graph_idx
        current_graph_idx = max(0, min(current_graph_idx, len(nodes_order) - 1))

        btn_graph_prev.disabled = (current_graph_idx == 0)
        btn_graph_next.disabled = (current_graph_idx == len(nodes_order) - 1)
        btn_graph_final.disabled = (current_graph_idx == len(nodes_order) - 1)

        visible_pipe_nodes = set(nodes_order[:current_graph_idx + 1])
        highlight_node = nodes_order[current_graph_idx]
        lbl_graph_step.value = f"Passo {current_graph_idx + 1} de {len(nodes_order)} | Nó: [{c['node_text_map_pipe'][highlight_node]}]"

        node_text_map_orig = {n: c['G_orig'].nodes[n]['text'] for n in c['G_orig'].nodes()}
        dot_orig = generate_graphviz_incremental(c['G_orig'], c['level_orig'], node_text_map_orig, None, set(c['G_orig'].nodes()))
        dot_pipe = generate_graphviz_incremental(c['G_pipe'], c['level_pipe'], c['node_text_map_pipe'], c['dest_changed_map_pipe'], visible_pipe_nodes, highlight_node, True)

        # Grafo completo do gabarito (todos os nós) para fins de exportação,
        # independente do passo atual da navegação
        dot_pipe_full = generate_graphviz_panel(c['G_pipe'], c['level_pipe'], c['node_text_map_pipe'], c['dest_changed_map_pipe'])

        out_l, out_r = widgets.Output(), widgets.Output()
        out_l.add_class('responsive-anim-graph')
        out_r.add_class('responsive-anim-graph')

        with out_l: display(dot_orig)
        with out_r: display(dot_pipe)

        title_style = 'color: #ffffff; text-align: center; font-weight: bold; font-family: sans-serif; font-size: 18px; margin: 0 0 15px 0; padding: 0;'

        # 📤 Botões de exportação próximos a cada respectivo grafo
        btn_export_orig2 = make_export_button(
            get_dot_fn=lambda: dot_orig, filename='grafo_original',
            feedback_output=pipe_compare_output, label='Exportar Original'
        )
        btn_export_gabarito = make_export_button(
            get_dot_fn=lambda: dot_pipe_full, filename='grafo_gabarito',
            feedback_output=pipe_compare_output, label='Exportar Gabarito'
        )

        col_l = widgets.VBox(
            [widgets.HTML(f'<h3 style="{title_style}">1. Grafo Original</h3>'),
             widgets.HBox([btn_export_orig2], layout=widgets.Layout(justify_content='center')),
             out_l],
            layout=widgets.Layout(width='50%')
        )
        col_r = widgets.VBox(
            [widgets.HTML(f'<h3 style="{title_style}">2. Gabarito</h3>'),
             widgets.HBox([btn_export_gabarito], layout=widgets.Layout(justify_content='center')),
             out_r],
            layout=widgets.Layout(width='50%')
        )
        split_box = widgets.HBox([col_l, col_r], layout=widgets.Layout(width='100%', align_items='flex-start'))
        display(split_box)

def on_graph_prev_clicked(b):
    global current_graph_idx
    if current_graph_idx > 0: current_graph_idx -= 1; render_current_graph_step()

def on_graph_next_clicked(b):
    global current_graph_idx
    if update_pipeline_cache() and current_graph_idx < len(pipeline_cache['nodes_order']) - 1:
        current_graph_idx += 1; render_current_graph_step()

def on_graph_final_clicked(b):
    global current_graph_idx
    if update_pipeline_cache(): current_graph_idx = len(pipeline_cache['nodes_order']) - 1; render_current_graph_step()

btn_graph_prev.on_click(on_graph_prev_clicked)
btn_graph_next.on_click(on_graph_next_clicked)
btn_graph_final.on_click(on_graph_final_clicked)

# =========================================================================
# 🗂️ MONTAGEM E INTEGRACÃO DA INTERFACE DE ABAS
# =========================================================================
row1_canvas = widgets.HBox([txt_node_instruction, slider_node_level, btn_add_canvas_node], layout=widgets.Layout(gap='15px'))
row2_canvas = widgets.HBox([txt_graph_code, btn_load_graph_code], layout=widgets.Layout(gap='15px'))
row3_canvas = widgets.HBox([drop_remove_node, btn_remove_canvas_node, btn_clear_canvas, btn_validate_canvas_graph], layout=widgets.Layout(gap='15px'))
panel_canvas_layout = widgets.VBox([
    widgets.VBox([row1_canvas, row2_canvas, row3_canvas], layout=widgets.Layout(padding='15px', background_color='#fafbfc', border='1px solid #cbd5e1', border_radius='12px', margin='10px 0')),
    canvas_output_plot,
    canvas_validation_output  # ✨ AQUI A SAÍDA É INSERIDA NA TELA DO COLAB
])

row1_gabarito = widgets.HBox([btn_graph_prev, btn_graph_next, btn_graph_final, lbl_graph_step], layout=widgets.Layout(gap='10px', margin='10px 0'))
panel_gabarito_layout = widgets.VBox([row1_gabarito, pipe_compare_output])

tab_interface = widgets.Tab()
tab_interface.children = [panel_canvas_layout, panel_gabarito_layout]
tab_interface.set_title(0, '✏️ Canvas Interativo')
tab_interface.set_title(1, '📖 Gabarito')

def on_tab_changed(change):
    if change['new'] == 1: render_current_graph_step()
    elif change['new'] == 0: render_canvas_split_view()

tab_interface.observe(on_tab_changed, names='selected_index')
dashboard_central = widgets.VBox([css_unificado, tab_interface])
display(dashboard_central)

update_remove_dropdown(); render_canvas_split_view()


### Gera código com soft. pipeline e original e compara a saída de ambos

In [47]:
#@title Usuário Propõe Código (Preâmbulo, Loop e Epílogo) e tem gabarito { display-mode: "form" }
# =========================================================================
# ⚙️ PATCH DE CORREÇÃO: Inicialização Real DO Banco Flutuante (F) e Fixes
# =========================================================================
def patched_run_full_program(self, instructions, R_init=None, F_init=None, mem=None, max_steps=500_000):
    R = dict(R_init) if R_init is not None else self.config.fresh_registers()
    F = dict(F_init) if F_init is not None else {k: v for k, v in self.config.fresh_registers().items() if k.startswith('f')}
    mem = mem if mem is not None else self.config.fresh_memory()
    labels = self.build_label_map(instructions)
    pc, steps = 0, 0
    label_visits = defaultdict(int)
    n = len(instructions)

    while 0 <= pc < n and steps < max_steps:
        instr = instructions[pc]
        steps += 1

        # 🔧 FIX 1: Contabiliza a visita ao rótulo PRIMEIRO
        if instr.get('label'):
            label_visits[instr['label']] += 1

        # Agora sim podemos pular se for apenas uma linha de rótulo
        if instr.get('type') == 'label':
            pc += 1
            continue

        cat = instr.get('category')
        if cat == 'branch':
            s1, s2, target = instr['operands']
            a, b, op = self._get_reg(s1, R, F), self._get_reg(s2, R, F), instr['op']
            taken = {'!=': a != b, '==': a == b, '<': a < b,
                     '>': a > b, '<=': a <= b, '>=': a >= b}[op]
            if taken and target in label_visits and label_visits[target] >= self.config.forced_loops:
                taken = False
            pc = labels[target] if taken else pc + 1
        elif cat == 'jump':
            (target,) = instr['operands']
            if target in label_visits and label_visits[target] >= self.config.forced_loops:
                pc = pc + 1
            else:
                pc = labels[target]
        else:
            self.execute_straightline(instr, R, F, mem)
            pc += 1

    return R, F, mem, label_visits

def patched_simulate_pipeline(self, stages_user, strides, total_iterations):
    ws = self.config.word_size
    mem_pipe = self.config.fresh_memory()
    F_pipe = {k: v for k, v in self.config.reg_init.items() if k.startswith('f')}
    max_lvl = max(stages_user.keys()) if stages_user else 0
    total_cycles = total_iterations + max_lvl

    # 🔧 NOVO: em vez do modelo linear "init + it*stride" (só funciona pra
    # dest==src), REPLICA fielmente as instruções i_type que atualizam
    # registradores inteiros, na ordem do programa original, uma vez por
    # iteração. Isso dá o valor REAL de cada registrador de endereço em
    # cada iteração, cobrindo indução (addi r1,r1,8) e reset (addi r1,r2,4).
    addr_update_instrs = [
        inst for inst in self.original_instructions
        if inst.get('category') == 'i_type' and not inst['operands'][0].startswith('f')
    ]
    R_by_iteration = []
    R_running = {k: v for k, v in self.config.reg_init.items() if not k.startswith('f')}
    for _ in range(total_iterations):
        R_by_iteration.append(dict(R_running))
        for inst in addr_update_instrs:
            dest, src, imm = inst['operands']
            a = R_running.get(src, 0)
            R_running[dest] = a + int(imm) if inst['op'] == '+' else a - int(imm)

    for cycle in range(total_cycles):
        F_snap = dict(F_pipe)
        mem_snap = list(mem_pipe)
        pending_F, pending_mem = {}, {}

        for lvl in sorted(stages_user.keys(), reverse=True):
            it = cycle - lvl
            if not (0 <= it < total_iterations):
                continue
            R_it = R_by_iteration[it]   # 🔧 substitui o cálculo linear antigo
            for instr in stages_user[lvl]:
                cat, ops, op = instr['category'], instr['operands'], instr['op']
                if cat == 'load':
                    dest, mem_op = ops
                    offset, base = MEM_RE.match(mem_op).groups()
                    addr = R_it.get(base, 0) + int(offset)
                    idx = addr // ws
                    if idx >= len(mem_pipe):
                        _ensure_mem_size(mem_pipe, idx)
                    val = mem_snap[idx] if idx < len(mem_snap) else float(idx)
                    pending_F[dest] = val
                elif cat == 'store':
                    src, mem_op = ops
                    offset, base = MEM_RE.match(mem_op).groups()
                    addr = R_it.get(base, 0) + int(offset)
                    idx = addr // ws
                    if idx >= len(mem_pipe):
                        _ensure_mem_size(mem_pipe, idx)
                    pending_mem[idx] = F_snap.get(src, pending_F.get(src, 0.0))
                elif cat == 'mov':
                    dest, src = ops
                    pending_F[dest] = F_snap.get(src, pending_F.get(src, 0.0))
                elif cat == 'r_type':
                    dest, s1, s2 = ops
                    a = F_snap.get(s1, pending_F.get(s1, 0.0))
                    b = F_snap.get(s2, pending_F.get(s2, 0.0))
                    pending_F[dest] = a * b if op == '*' else (a + b if op == '+' else a - b)
                elif cat == 'i_type':
                    dest, src, imm = ops
                    a = F_snap.get(src, pending_F.get(src, 0.0)) if src.startswith('f') else R_it.get(src, 0)
                    pending_F[dest] = a + int(imm) if op == '+' else a - int(imm)

        F_pipe.update(pending_F)
        for idx, val in pending_mem.items():
            mem_pipe[idx] = val
    return mem_pipe

PipelineValidator.simulate_pipeline = patched_simulate_pipeline

# 🔧 FIX 2: Usar .get('category') para evitar KeyError em rótulos isolados
def patched_detect_induction_strides(instructions):
    strides = {}
    for instr in instructions:
        if instr.get('category') == 'i_type':
            dest, src, imm = instr['operands']
            if dest.startswith('f'):
                continue
            if dest == src:
                # Caso clássico de indução: rX = rX + stride
                strides[dest] = int(imm) if instr['op'] == '+' else -int(imm)
            else:
                # 🔧 NOVO: dest != src -> não é indução, é um RESET para valor fixo.
                # O endereço associado a "dest" não avança entre iterações (stride = 0).
                strides[dest] = 0
    return strides

ISASimUniform = patched_run_full_program
ISASimulator.run_full_program = patched_run_full_program
ISASimulator.detect_induction_strides = staticmethod(patched_detect_induction_strides)
PipelineValidator.simulate_pipeline = patched_simulate_pipeline


# =========================================================================
# 📦 GERADORES DE GRAFO E AUXILIARES (Definidos antes da chamada inicial)
# =========================================================================
def extract_dest_reg_local(text):
    text = text.split('#')[0].strip().lower()
    parts = text.split(None, 1)
    if len(parts) < 2: return None
    opcode, rest = parts
    if opcode in ['sd', 'sw', 'sb', 'sh', 'bne', 'beq', 'bgt', 'blt']:
        return None
    subparts = rest.split(',')
    if subparts:
        possible_reg = subparts[0].strip()
        if re.match(r'^f\d+$', possible_reg):
            return possible_reg
    return None

def build_pipelined_with_moves(G_orig, level_orig):
    all_regs = []
    for node in G_orig.nodes():
        all_regs.extend(re.findall(r'[fF]\d+', G_orig.nodes[node]['text']))
    reg_indices = [int(r[1:]) for r in all_regs if r[0].lower() == 'f']
    max_reg_idx = max(reg_indices) if reg_indices else 0
    global_next_reg = max_reg_idx + 1

    G_pipe = nx.DiGraph()
    level_pipe, node_text_map, dest_changed_map = {}, {}, {}
    levels_to_nodes = defaultdict(list)
    for node, lvl in level_orig.items():
        levels_to_nodes[lvl].append(node)

    reg_name_at_level = defaultdict(dict)
    mov_nodes, move_node_counter, seen_destinations = {}, 1000, set()

    prod_consumers = defaultdict(list)
    for u, v, data in G_orig.edges(data=True):
        for r in data['regs']:
            prod_consumers[(u, r.lower())].append(v)

    for u in G_orig.nodes():
        orig_text = G_orig.nodes[u]['text']
        d = extract_dest_reg_local(orig_text)
        if d: reg_name_at_level[(u, d.lower())][level_orig[u]] = d

    prefer_upper = any(r.isupper() for r in all_regs)
    f_prefix = 'F' if prefer_upper else 'f'
    mov_opcode = 'MOV' if prefer_upper else 'mov'

    for lvl in sorted(levels_to_nodes.keys()):
        for (p, r), consumers in list(prod_consumers.items()):
            L_p = level_orig[p]
            L_end = max(level_orig[c] for c in consumers)
            if L_p < lvl <= L_end:
                if lvl < L_end:
                    mov_id = move_node_counter
                    move_node_counter += 1
                    r_prev = reg_name_at_level[(p, r.lower())][lvl - 1]
                    r_new = f"{f_prefix}{global_next_reg}"
                    global_next_reg += 1

                    node_text_map[mov_id] = f"{mov_opcode} {r_new}, {r_prev}"
                    level_pipe[mov_id] = lvl
                    dest_changed_map[mov_id] = True
                    G_pipe.add_node(mov_id)
                    mov_nodes[(p, r.lower(), lvl)] = mov_id
                    reg_name_at_level[(p, r.lower())][lvl] = r_new
                    parent = p if lvl - 1 == L_p else mov_nodes[(p, r.lower(), lvl - 1)]
                    G_pipe.add_edge(parent, mov_id, regs={r_prev})
                else:
                    reg_name_at_level[(p, r.lower())][lvl] = reg_name_at_level[(p, r.lower())][lvl - 1]

        for u in levels_to_nodes[lvl]:
            orig_text = G_orig.nodes[u]['text']
            src_mappings = {}
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    src_mappings[r.lower()] = reg_name_at_level[(p_node, r.lower())][lvl - 1]

            orig_dest = extract_dest_reg_local(orig_text)
            new_dest = orig_dest
            dest_changed = False
            if orig_dest:
                orig_dest_lower = orig_dest.lower()
                if orig_dest_lower in seen_destinations:
                    new_dest = f"{f_prefix}{global_next_reg}"
                    global_next_reg += 1
                    dest_changed = True
                seen_destinations.add(orig_dest_lower)

            parts = orig_text.split(None, 1)
            if len(parts) == 2:
                opcode, rest = parts
                subparts = rest.split(',')
                is_store = opcode.lower() in ['sd', 'sw', 'sb', 'sh']
                new_subparts = []
                for idx, sub in enumerate(subparts):
                    sub_clean = sub.strip()
                    if idx == 0 and not is_store and orig_dest is not None:
                        new_subparts.append(new_dest)
                    else:
                        updated_sub = sub_clean
                        for orig_src, mapped_name in src_mappings.items():
                            updated_sub = re.sub(r'\b' + re.escape(orig_src) + r'\b', mapped_name, updated_sub, flags=re.IGNORECASE)
                        new_subparts.append(updated_sub)

                if opcode.lower() == 'mv':
                    opcode = mov_opcode

                updated_text = f"{opcode} {', '.join(new_subparts)}"
            else:
                updated_text = orig_text

            node_text_map[u] = updated_text
            level_pipe[u] = lvl
            dest_changed_map[u] = dest_changed
            G_pipe.add_node(u)

            if orig_dest: reg_name_at_level[(u, orig_dest.lower())][lvl] = new_dest
            for p_node in G_orig.predecessors(u):
                for r in G_orig[p_node][u]['regs']:
                    parent_in_pipe = p_node if lvl - 1 == level_orig[p_node] else mov_nodes[(p_node, r.lower(), lvl - 1)]
                    G_pipe.add_edge(parent_in_pipe, u, regs={reg_name_at_level[(p_node, r.lower())][lvl - 1]})

    return G_pipe, level_pipe, node_text_map, dest_changed_map


# =========================================================================
# 🔧 FUNÇÃO COMPARTILHADA: forma do pipeline (usada pela Aba 1 e Aba 2)
# =========================================================================
def normalize_mem_instruction(text):
    text = text.split('#')[0].strip().lower()
    text_normalized = re.sub(r'(-?\d+)\((\w+)\)', r'(\2)', text)
    text_normalized = re.sub(r'\s*,\s*', ',', text_normalized)
    text_normalized = re.sub(r'\s+', ' ', text_normalized)
    return text_normalized

def _compute_pipeline_shape():
    """Retorna (max_level, signature_to_level, stride) a partir de program_instructions."""
    if 'program_instructions' not in globals() or not program_instructions:
        return 0, {}, 8
    G_orig = build_dependency_graph(program_instructions)
    no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
    G_orig.remove_nodes_from(no_dep)
    level_orig = compute_layered_positions(G_orig)
    _, level_pipe, node_text_map_pipe, _ = build_pipelined_with_moves(G_orig, level_orig)
    max_level = max(level_pipe.values()) if level_pipe else 0

    signature_to_level = {}
    for node, lvl in level_pipe.items():
        sig = normalize_mem_instruction(node_text_map_pipe[node])
        signature_to_level[sig] = lvl

    strides = ISASimulator.detect_induction_strides(program_instructions)
    base_reg = 'r1'
    for inst in program_instructions:
        if inst.get('category') in ('load', 'store'):
            ops = inst.get('operands', [])
            if len(ops) == 2:
                mm = re.search(r'\((r\d+)\)', ops[1], flags=re.IGNORECASE)
                if mm:
                    base_reg = mm.group(1).lower()
                    break

    # 🔧 NOVO: sem fallback silencioso de 8. Se não detectou indução,
    # trata como stride 0 (endereço constante) e avisa o usuário.
    if base_reg not in strides:
        print(f"⚠️ AVISO: nenhuma variável de indução detectada para '{base_reg}'. "
              f"Assumindo stride=0 (endereço fixo). Verifique se seu 'addi' usa "
              f"o formato 'addi {base_reg}, {base_reg}, imm'.")
    stride = strides.get(base_reg, 0)
    return max_level, signature_to_level, stride


# =========================================================================
# ✏️ ABA 1 — WIDGETS DO SANDBOX (um campo de texto por ciclo, sem tags)
# =========================================================================
challenge_title = widgets.HTML("""
    <div style="background: linear-gradient(135deg, #2e1065 0%, #3b0764 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;"> Valide seu Próprio Pipeline</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha sua escala de código.</p>
    </div>
""")

_MAX_LEVEL, _SIG_TO_LEVEL, _STRIDE = _compute_pipeline_shape()
_N_PREAMBLE_CYCLES = _MAX_LEVEL
_N_EPILOGUE_CYCLES = _MAX_LEVEL

def _make_cycle_box(label):
    return widgets.Textarea(
        value='',
        placeholder='Instruções ativas neste ciclo...',
        description=label,
        layout=widgets.Layout(width='98%', height='70px', margin='0 0 6px 0'),
        style={'description_width': '160px'}
    )

preamble_cycle_boxes = [
    _make_cycle_box(f'Iteração {k+1}:')
    for k in range(_N_PREAMBLE_CYCLES)
] or [_make_cycle_box('Sem preâmbulo necessário:')]

user_kernel_input = widgets.Textarea(
    value='# Digite seu miolo do loop aqui\n',
    placeholder='Instruções do Loop...',
    description='Loop:',
    layout=widgets.Layout(width='98%', height='100px', margin='10px 0 10px 0')
)

epilogue_cycle_boxes = [
    _make_cycle_box(f'Iteração {_N_PREAMBLE_CYCLES + k + 1}:')
    for k in range(_N_EPILOGUE_CYCLES)
] or [_make_cycle_box('Sem epílogo necessário:')]

btn_validate_challenge = widgets.Button(
    description='Validar Equivalência Semântica', icon='check-double',
    button_style='primary', layout=widgets.Layout(width='320px', height='42px')
)

challenge_output = widgets.Output(layout=widgets.Layout(margin='16px 0 0 0'))

def _translate_lines(lines_text, current_cycle, signature_to_level, stride):
    out = []
    for line in lines_text.split('\n'):
        clean_content = line.split('#')[0].strip()
        if not clean_content:
            continue
        sig = normalize_mem_instruction(clean_content)
        if sig in signature_to_level:
            lvl = signature_to_level[sig]
            it = current_cycle - lvl
            mm_off = re.search(r'(-?\d+)\(\w+\)', clean_content)
            if mm_off:
                orig_offset = int(mm_off.group(1))
                adj_offset = orig_offset + it * stride
                clean_content = re.sub(
                    r'-?\d+(\(\w+\))', rf'{adj_offset}\1', clean_content, flags=re.IGNORECASE
                )
        out.append(clean_content)
    return out

def on_validate_challenge_clicked(b):
    with challenge_output:
        clear_output()

        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Código base não encontrado. Certifique-se de carregar as instruções na Célula 2.")
            return

        validator_oficial = PipelineValidator(program_instructions, machine_config)
        try:
            mem_gabarito, _, total_iterations = validator_oficial.compute_reference()
        except Exception as e:
            print(f"❌ Erro ao rodar o programa de referência: {e}")
            return

        max_level, signature_to_level, stride = _compute_pipeline_shape()

        pre_lines = []
        for k, box in enumerate(preamble_cycle_boxes):
            pre_lines.extend(_translate_lines(box.value, k, signature_to_level, stride))

        print(preamble_cycle_boxes)

        ker_lines = []
        for line in user_kernel_input.value.split('\n'):
            clean_content = line.split('#')[0].strip()
            if not clean_content:
                continue
            sig = normalize_mem_instruction(clean_content)
            if sig in signature_to_level:
                lvl = signature_to_level[sig]
                mm_off = re.search(r'(-?\d+)\(\w+\)', clean_content)
                if mm_off:
                    orig_offset = int(mm_off.group(1))
                    adj_offset = orig_offset + (max_level - lvl) * stride
                    clean_content = re.sub(
                        r'-?\d+(\(\w+\))', rf'{adj_offset}\1', clean_content, flags=re.IGNORECASE
                    )
            ker_lines.append(clean_content)

        epi_lines = []
        for k, box in enumerate(epilogue_cycle_boxes):
            current_e_cycle = k + 1
            for line in box.value.split('\n'):
                clean_content = line.split('#')[0].strip()
                if not clean_content:
                    continue
                sig = normalize_mem_instruction(clean_content)
                if sig in signature_to_level:
                    lvl = signature_to_level[sig]
                    mm_off = re.search(r'(-?\d+)\(\w+\)', clean_content)
                    if mm_off:
                        orig_offset = int(mm_off.group(1))
                        adj_offset = orig_offset + (max_level - lvl + current_e_cycle - 1) * stride
                        clean_content = re.sub(
                            r'-?\d+(\(\w+\))', rf'{adj_offset}\1', clean_content, flags=re.IGNORECASE
                        )
                epi_lines.append(clean_content)

        has_loop_control = any(
            ':' in line or any(op in line for op in ['bne', 'beq', 'bge', 'ble', 'bgt', 'blt', 'j'])
            for line in ker_lines
        )

        if has_loop_control:
            user_program_text = "\n".join(pre_lines + ker_lines + epi_lines)
        else:
            user_program_text = "\n".join(pre_lines + (ker_lines * total_iterations) + epi_lines)

        try:
            user_parsed_instructions = parse_program(user_program_text)

            # =========================================================================
            # 🔧 PATCH DE CORREÇÃO DEFINITIVO: Ajuste de limites do Loop (Sandbox)
            # =========================================================================
            sandbox_config = MachineConfig(
                mem_size=machine_config.mem_size,
                word_size=machine_config.word_size,
                forced_loops=machine_config.forced_loops
            )

            R_init_sandbox = dict(sandbox_config.reg_init)

            if has_loop_control:
                sandbox_config.forced_loops = max(1, total_iterations - max_level)

                for inst in user_parsed_instructions:
                    if inst.get('category') == 'branch':
                        rA, rB = inst['operands'][0], inst['operands'][1]
                        valA = R_init_sandbox.get(rA, 0)
                        valB = R_init_sandbox.get(rB, 0)

                        if valA > valB:
                            R_init_sandbox[rA] -= (max_level * stride)
                        elif valB > valA:
                            R_init_sandbox[rB] -= (max_level * stride)
                        break

            simulador_sandbox = ISASimulator(sandbox_config)
            F_init_sandbox = {k: v for k, v in sandbox_config.reg_init.items() if k.startswith('f')}
            # =========================================================================

            _, _, mem_usuario, _ = simulador_sandbox.run_full_program(
                user_parsed_instructions,
                R_init=R_init_sandbox,
                F_init=F_init_sandbox,
            )
        except Exception as e:
            print(f"❌ Erro de parsing ou execução no seu código proposto: {e}")
            return

        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        mismatches_count = 0
        for i in range(len(mem_gabarito)):
            if mem_gabarito[i] != mem_usuario[i]:
                mismatches_count += 1

        table_rows = ""
        n_show = min(30, machine_config.mem_size)
        for i in range(n_show):
            v_init = float(i)
            v_gab = mem_gabarito[i]
            v_usr = mem_usuario[i]

            is_modified = (v_gab != v_init)
            row_bg = "background-color: #020617;" if is_modified else ""
            status = "✔ Correto" if v_gab == v_usr else "❌ Erro"
            status_color = "#4ade80" if v_gab == v_usr else "#f87171"

            table_rows += f"""
            <tr style="{row_bg}">
                <td style="{td_style} color: #94a3b8; font-weight: bold;">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init:.1f}</td>
                <td style="{td_style} color: #38bdf8;">{v_gab:.1f}</td>
                <td style="{td_style} color: #e2e8f0;">{v_usr:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        if mismatches_count == 0:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #064e3b; border-left: 4px solid #10b981; color: #34d399; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                🎉 <b>Veredicto: PIPELINE APROVADO!</b> Seu arranjo de instruções respeitou todas as dependências temporais. O resultado final gerado na simulação é perfeitamente equivalente.
            </div>
            """
        else:
            veredicto_html = f"""
            <div style="margin-top: 16px; padding: 16px; background-color: #4c0519; border-left: 4px solid #f43f5e; color: #f43f5e; border-radius: 6px; font-family: sans-serif; font-size: 14px;">
                ❌ <b>Veredicto: FALHA DE EQUIVALÊNCIA!</b> O simulador detectou {mismatches_count} divergências de dados na memória física. Revise a ordenação das instruções.
            </div>
            """

        html_view = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 18px; margin-bottom: 12px;">📊 Verificação de Impacto no Vetor de Dados (Configuração Ativa)</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 400px; overflow-y: auto; width: 100%;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Vetor</th>
                            <th style="{th_style}">Estado Inicial</th>
                            <th style="{th_style}">Gabarito Sequencial</th>
                            <th style="{th_style}">Seu Pipeline</th>
                            <th style="{th_style}">Resultado</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            {veredicto_html}
        """, layout=widgets.Layout(width='100%'))

        display(html_view)

btn_validate_challenge.on_click(on_validate_challenge_clicked)

def _section_header(title, color, subtitle=""):
    return widgets.HTML(f"""
        <div style="display:flex; align-items:baseline; gap:10px; margin:18px 0 8px 0;
                    padding-left:12px; border-left:4px solid {color};">
            <span style="font-family:'Segoe UI',sans-serif; font-size:17px; font-weight:700;
                         color:#e2e8f0; letter-spacing:0.3px;">{title}</span>
            {f'<span style="font-family:sans-serif; font-size:12px; color:#94a3b8;">{subtitle}</span>' if subtitle else ''}
        </div>
    """)

form_box = widgets.VBox([
    challenge_title,
    widgets.VBox(
        [_section_header("Preâmbulo", "#f59e0b")]
        + preamble_cycle_boxes
        + [_section_header("Loop", "#10b981")]
        + [user_kernel_input]
        + [_section_header("Epílogo", "#ef4444")]
        + epilogue_cycle_boxes
        + [widgets.HBox([btn_validate_challenge], layout=widgets.Layout(align_items='center', margin='16px 0 0 0'))],
        layout=widgets.Layout(padding='20px', border='1px solid #334155',
                              border_radius='0 0 12px 14px', background_color='#0f172a')
    )
])


# =========================================================================
# 📖 ABA 2 — WIDGETS DO GABARITO (código gerado automaticamente)
# =========================================================================
asm_compare_button = widgets.Button(
    description='Gerar Código e Simular', icon='play',
    button_style='success', layout=widgets.Layout(width='380px', height='40px')
)
asm_compare_output = widgets.Output(layout=widgets.Layout(margin='14px 0 0 0'))

def generate_asm_text_blocks(level_pipe, node_text_map_pipe, program_instructions):
    if not level_pipe:
        vazio = "    # (nenhuma instrução escalonada)"
        return vazio, vazio, vazio

    max_level = max(level_pipe.values())

    stages = defaultdict(list)
    for node, lvl in level_pipe.items():
        adjusted_text = node_text_map_pipe[node]
        stages[lvl].append(adjusted_text)

    def block_for_levels(levels_subset, ascending=False):
        lines = []
        order = sorted(levels_subset, reverse=not ascending)
        for lvl in order:
            lines.extend(stages.get(lvl, []))
        return lines

    ctrl_instructions = [
        inst['raw'] for inst in program_instructions
        if inst.get('category') in ('branch', 'jump') or
        (inst.get('category') == 'i_type' and not inst['operands'][0].startswith('f'))
    ]
    label_name = next((inst['label'] for inst in program_instructions if inst.get('label')), None) or 'LOOP'

    preamble_lines = []
    for k in range(max_level):
        preamble_lines.append(f"    # Iteração {k + 1}:")
        preamble_lines.extend(f"    {line}" for line in block_for_levels(range(0, k + 1)))
        preamble_lines.append("")
    preamble_text = "\n".join(preamble_lines).rstrip("\n") if preamble_lines else "    # (pipeline raso — sem preâmbulo necessário)"

    kernel_lines = [f"{label_name}:"]
    kernel_lines.extend(f"    {line}" for line in block_for_levels(stages.keys()))
    kernel_lines.extend(f"    {ctrl}" for ctrl in ctrl_instructions)
    kernel_text = "\n".join(kernel_lines)

    epilogue_lines = []
    for k in range(1, max_level + 1):
        epilogue_lines.append(f"    # Iteração {max_level + k}:")
        epilogue_lines.extend(f"    {line}" for line in block_for_levels(range(k, max_level + 1)))
        epilogue_lines.append("")
    epilogue_text = "\n".join(epilogue_lines).rstrip("\n") if epilogue_lines else "    # (pipeline raso — sem epílogo necessário)"

    return preamble_text, kernel_text, epilogue_text

def on_asm_compare_clicked(b):
    with asm_compare_output:
        clear_output()

        if 'program_instructions' not in globals() or not program_instructions:
            print("⚠️ Nenhuma instrução encontrada no programa. Adicione instruções na primeira célula.")
            return

        G_orig = build_dependency_graph(program_instructions)
        no_dep = [n for n in G_orig.nodes() if G_orig.degree(n) == 0]
        G_orig.remove_nodes_from(no_dep)

        if G_orig.number_of_nodes() == 0:
            print("ℹ️ Nenhuma dependência de registradores para processar.")
            return

        level_orig = compute_layered_positions(G_orig)
        G_pipe, level_pipe, node_text_map_pipe, dest_changed_map_pipe = build_pipelined_with_moves(G_orig, level_orig)

        pre_text, kernel_text, epi_text = generate_asm_text_blocks(level_pipe, node_text_map_pipe, program_instructions)

        # 🔧 FIX 3: Construir o texto original direto das instruções reais (exibe o addi e o bne)
        orig_insts_list = []
        has_label = False
        for inst in program_instructions:
            raw = inst.get('raw', '')
            if inst.get('type') == 'label':
                orig_insts_list.append(raw)
                has_label = True
            else:
                orig_insts_list.append(f"    {raw}")

        orig_text = ("" if has_label else "LOOP_ORIGINAL:\n") + "\n".join(orig_insts_list)

        generated_pipeline_nodes = [{'text': node_text_map_pipe[n], 'level': level_pipe[n]} for n in G_pipe.nodes()]

        validator = PipelineValidator(program_instructions, machine_config)
        result = validator.validate(generated_pipeline_nodes)

        if result.get("error"):
            print(f"❌ {result['error']}")
            return

        code_style = (
            "background-color: #0f172a; color: #38bdf8; padding: 16px; "
            "border-radius: 10px; font-family: 'Consolas', monospace; "
            "font-size: 14px; line-height: 1.6; overflow-x: auto; white-space: pre;"
        )

        html_original = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🔄 Código Original</h4>
            <div style="{code_style} color: #e2e8f0;">{orig_text}</div>
        """, layout=widgets.Layout(width='35%'))

        html_pipelined = widgets.HTML(f"""
            <h4 style="color: #f1f5f9; font-family: sans-serif; margin-bottom: 8px; font-weight: 600; font-size: 20px;">🚀 Código com Software Pipeline</h4>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #f59e0b; color: #fbbf24;">{pre_text}</div>
            <div style="{code_style} margin-bottom: 12px; border-left: 4px solid #10b981; color: #34d399;">{kernel_text}</div>
            <div style="{code_style} border-left: 4px solid #ef4444; color: #f87171;">{epi_text}</div>
        """, layout=widgets.Layout(width='62%'))

        layout_codigo = widgets.HBox([html_original, html_pipelined], layout=widgets.Layout(gap='20px', width='100%'))

        th_style = "padding: 10px; background-color: #1e293b; color: #f1f5f9; position: sticky; top: 0; font-size: 13px;"
        td_style = "padding: 8px 10px; border-bottom: 1px solid #1e293b; font-family: monospace; font-size: 13px;"

        table_rows = ""
        mem_gabarito = result["mem_gabarito"]
        mem_pipe = result["mem_pipe"]
        n_show = min(30, machine_config.mem_size)

        for i in range(n_show):
            v_init = float(i)
            v_mem = mem_gabarito[i]
            v_l = mem_pipe[i]

            is_modified = (v_mem != v_init)
            row_bg = "background-color: #111827;" if is_modified else ""
            idx_style = "color: #38bdf8; font-weight: bold;" if is_modified else "color: #94a3b8;"
            status = "✔ Perfeito" if v_mem == v_l else "❌ Mismatch"
            status_color = "#4ade80" if v_mem == v_l else "#f87171"

            table_rows += f"""
            <tr style="{row_bg} border-bottom: 1px solid #1e293b;">
                <td style="{td_style} {idx_style}">Índice [{i}]</td>
                <td style="{td_style} color: #64748b;">{v_init}</td>
                <td style="{td_style} color: #38bdf8;">{v_mem:.1f}</td>
                <td style="{td_style} color: #34d399;">{v_l:.1f}</td>
                <td style="{td_style} color: {status_color}; font-weight: bold;">{status}</td>
            </tr>
            """

        status_veredicto = "✔ <b>Sucesso na Homologação:</b> O estado final bateu 100% idêntico!" if result["success"] else "❌ <b>Falha na Homologação:</b> Ocorreu divergência matemática."
        bg_veredicto = "#14532d" if result["success"] else "#4c0519"
        color_veredicto = "#4ade80" if result["success"] else "#f43f5e"

        html_tables = widgets.HTML(f"""
            <hr style="border: 0; border-top: 1px solid #334155; margin: 24px 0;">
            <h4 style="color: #f1f5f9; font-family: sans-serif; font-weight: 600; font-size: 20px; margin-bottom: 4px;">📊 Comparação dos valores da memória: Código Original vs. Soft. Pipel.</h4>
            <div style="background-color: #0f172a; padding: 12px; border-radius: 8px; max-height: 450px; overflow-y: auto; width: 100%;">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-family: sans-serif;">
                    <thead>
                        <tr>
                            <th style="{th_style}">Posição do Vetor</th>
                            <th style="{th_style}">Valor Inicial</th>
                            <th style="{th_style}">Loop Original (Gabarito)</th>
                            <th style="{th_style}">Código Otimizado Automático</th>
                            <th style="{th_style}">Status de Validação</th>
                        </tr>
                    </thead>
                    <tbody>{table_rows}</tbody>
                </table>
            </div>
            <div style="margin-top: 16px; padding: 14px; background-color: {bg_veredicto}; color: {color_veredicto}; border-radius: 6px; font-family: sans-serif; font-size: 14px; font-weight: 500;">
                {status_veredicto}
            </div>
        """, layout=widgets.Layout(width='100%'))

        display(widgets.VBox([layout_codigo, html_tables]))

asm_compare_button.on_click(on_asm_compare_clicked)
asm_compare_output.add_class('output-box')


# =========================================================================
# 🗂️ MONTAGEM FINAL: AS 2 ABAS, UMA ÚNICA CÉLULA, UM ÚNICO display()
# =========================================================================
css_unificado_asm = widgets.HTML("""
    <style>
    .lm-TabBar-tab, .p-TabBar-tab {
        width: auto !important; min-width: 220px !important;
        padding: 0 20px !important; overflow: visible !important;
    }
    .lm-TabBar-tabLabel, .p-TabBar-tabLabel {
        color: #ffffff !important; font-weight: 600 !important;
        font-size: 13px !important; overflow: visible !important; text-overflow: clip !important;
    }
    .lm-TabBar-tab:not(.lm-mod-current), .p-TabBar-tab:not(.p-mod-current) {
        background-color: #334155 !important; opacity: 0.7;
    }
    .lm-TabBar-tab.lm-mod-current, .p-TabBar-tab.p-mod-current {
        background-color: #1e293b !important; border-bottom: 2px solid #38bdf8 !important; opacity: 1;
    }

    /* ---- Editor estilo VSCode para os Textareas de código (Preâmbulo/Kernel/Epílogo) ---- */
    .widget-textarea textarea {
        font-family: 'Cascadia Code', 'Fira Code', 'JetBrains Mono', Consolas, 'Courier New', monospace !important;
        font-size: 13px !important;
        line-height: 1.6 !important;
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
        border: 1px solid #3c3c3c !important;
        border-radius: 6px !important;
        padding: 10px 12px !important;
        caret-color: #ffffff !important;
        tab-size: 4 !important;
    }
    .widget-textarea textarea:focus {
        border-color: #007acc !important;
        outline: none !important;
        box-shadow: 0 0 0 1px #007acc !important;
    }
    .widget-textarea textarea::selection {
        background-color: #264f78 !important;
    }
    .widget-textarea textarea::placeholder {
        color: #6a6a6a !important;
        font-style: italic;
    }
    .widget-textarea .widget-label {
        font-family: 'Cascadia Code', Consolas, monospace !important;
        color: #9cdcfe !important;
        font-weight: 600 !important;
        font-size: 12.5px !important;
    }
    .widget-textarea {
        margin-bottom: 2px !important;
    }
    .widget-textarea textarea {
        box-shadow: inset 0 1px 3px rgba(0,0,0,0.3) !important;
        transition: border-color 0.15s ease;
    }
    .widget-textarea textarea:hover:not(:focus) {
        border-color: #5a5a5a !important;
    }
    .widget-textarea textarea::-webkit-scrollbar {
        width: 10px !important;
        height: 10px !important;
    }
    .widget-textarea textarea::-webkit-scrollbar-track {
        background: #1e1e1e !important;
    }
    .widget-textarea textarea::-webkit-scrollbar-thumb {
        background-color: #4a4a4a !important;
        border-radius: 5px !important;
        border: 2px solid #1e1e1e !important;
    }
    .widget-textarea textarea::-webkit-scrollbar-thumb:hover {
        background-color: #6a6a6a !important;
    }
    </style>
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); padding: 18px 24px; border-radius: 12px 14px 0 0; color: white; font-family: sans-serif;">
        <h3 style="margin: 0; font-size: 22px; font-weight: 600;">👨‍💻 Central de Código: Preâmbulo, Loop e Epílogo</h3>
        <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.85;">Proponha seu próprio código escalonado no editor ou veja o Gabarito gerado automaticamente a partir do grafo.</p>
    </div>
""")

panel_sandbox_layout = widgets.VBox([form_box, challenge_output])
panel_gabarito_asm_layout = widgets.VBox([
    widgets.HBox([asm_compare_button], layout=widgets.Layout(margin='10px 0')),
    asm_compare_output
])

tab_interface_asm = widgets.Tab()
tab_interface_asm.children = [panel_sandbox_layout, panel_gabarito_asm_layout]
tab_interface_asm.set_title(0, '✏️ Proponha seu Código')
tab_interface_asm.set_title(1, '📖 Gabarito')

def on_tab_changed_asm(change):
    if change['new'] == 1:
        on_asm_compare_clicked(None)

tab_interface_asm.observe(on_tab_changed_asm, names='selected_index')

dashboard_central_asm = widgets.VBox([css_unificado_asm, tab_interface_asm])
display(dashboard_central_asm)

### Visualize os diferentes escalonamentos no pipeline do processador

In [63]:
#@title Visualize a simulação
app = SimulatorController()

Dropdown(description='Escalonamento:', layout=Layout(margin='5px 0px', width='100%'), options=(('Código Origin…

HTML(value="<div style='border: 1px solid rgba(128,128,128,0.3); padding: 10px; border-radius: 4px; background…

Button(button_style='primary', description='Iniciar Comparação Lado a Lado', layout=Layout(margin='15px 0px', …

HTML(value='')

Output()